# Notebook 08 — Final Model Training & Kaggle Submission

## Objective

This notebook trains the final selected machine learning solution using the complete training dataset and generates the official Kaggle submission for the ROGII Wellbore Geology Prediction competition.

Unlike previous notebooks, this notebook does not perform model comparison, validation, or hyperparameter optimization. Those activities were completed in earlier stages of the project.

The final solution selected from Notebook 07 is a weighted ensemble consisting of:

- Extra Trees Regressor (80%)
- XGBoost Regressor (20%)

The notebook performs the following tasks:

1. Load the complete training dataset.
2. Reconstruct the engineered feature set.
3. Train the final Extra Trees model.
4. Train the final XGBoost model.
5. Generate predictions for the Kaggle test wells.
6. Apply the weighted ensemble.
7. Create the official `submission.csv`.
8. Save the trained models and project artifacts.

### 1. Import Libraries

In [88]:
from pathlib import Path
import json
import sys
import time
import warnings

import joblib
import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

### 2. Configure Project Paths

In [89]:
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

TRAIN_DIR = RAW_DATA_DIR / "train"
TEST_DIR = RAW_DATA_DIR / "test"

RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

NOTEBOOK_07B_RESULTS_DIR = (
    RESULTS_DIR
    / "notebook_07b"
)

NOTEBOOK_08_RESULTS_DIR = (
    RESULTS_DIR
    / "notebook_08"
)

NOTEBOOK_08_MODELS_DIR = (
    MODELS_DIR
    / "notebook_08"
)

SRC_DIR = PROJECT_ROOT / "src"

# Add the project root so imports such as
# `from src.feature_engineering import ...` work correctly.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NOTEBOOK_08_RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NOTEBOOK_08_MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root:", PROJECT_ROOT)
print("Source directory:", SRC_DIR)
print("Source directory exists:", SRC_DIR.exists())
print("Training directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)
print("Notebook 08 results:", NOTEBOOK_08_RESULTS_DIR)
print("Notebook 08 models:", NOTEBOOK_08_MODELS_DIR)

Project root: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology
Source directory: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\src
Source directory exists: True
Training directory: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\data\raw\train
Test directory: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\data\raw\test
Notebook 08 results: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08
Notebook 08 models: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\models\notebook_08


### 3. Discover and Load Full Training Data

This section discovers all paired training wells and verifies that the complete training dataset is available before final model training.

In [90]:
train_csv_files = sorted(TRAIN_DIR.rglob("*.csv"))

print(f"Number of training CSV files: {len(train_csv_files)}")

for path in train_csv_files[:10]:
    print(path.relative_to(TRAIN_DIR))

Number of training CSV files: 1546
000d7d20__horizontal_well.csv
000d7d20__typewell.csv
00bbac68__horizontal_well.csv
00bbac68__typewell.csv
00e12e8b__horizontal_well.csv
00e12e8b__typewell.csv
015fe0d2__horizontal_well.csv
015fe0d2__typewell.csv
01869cd4__horizontal_well.csv
01869cd4__typewell.csv


In [91]:
horizontal_train_files = sorted(
    TRAIN_DIR.rglob("*horizontal*.csv")
)

typewell_train_files = sorted(
    TRAIN_DIR.rglob("*typewell*.csv")
)

print("Horizontal training files:", len(horizontal_train_files))
print("Typewell training files:", len(typewell_train_files))

Horizontal training files: 773
Typewell training files: 773


In [92]:
assert len(horizontal_train_files) > 0, (
    "No horizontal training files were found."
)

assert len(typewell_train_files) > 0, (
    "No typewell training files were found."
)

assert len(horizontal_train_files) == len(typewell_train_files), (
    "The number of horizontal and typewell files does not match."
)

print(
    f"Successfully discovered "
    f"{len(horizontal_train_files)} training well pairs."
)

Successfully discovered 773 training well pairs.


### 4. Discover and Inspect Kaggle Test Data

The Kaggle test data contains the wells for which final TVT predictions must be generated.

The hidden target values are not available. Therefore, this section only verifies the test files, identifies the required submission rows, and prepares the test wells for feature construction.

In [93]:
test_csv_files = sorted(TEST_DIR.rglob("*.csv"))

print(f"Number of test CSV files: {len(test_csv_files)}")

for path in test_csv_files:
    print(path.relative_to(TEST_DIR))

Number of test CSV files: 7
000d7d20__horizontal_well.csv
000d7d20__typewell.csv
00bbac68__horizontal_well.csv
00bbac68__typewell.csv
00e12e8b__horizontal_well.csv
00e12e8b__typewell.csv
sample_submission.csv


In [94]:
horizontal_test_files = sorted(
    TEST_DIR.rglob("*horizontal*.csv")
)

typewell_test_files = sorted(
    TEST_DIR.rglob("*typewell*.csv")
)

print("Horizontal test files:", len(horizontal_test_files))
print("Typewell test files:", len(typewell_test_files))

Horizontal test files: 3
Typewell test files: 3


In [95]:
assert len(horizontal_test_files) > 0, (
    "No horizontal test files were found."
)

assert len(typewell_test_files) > 0, (
    "No typewell test files were found."
)

assert len(horizontal_test_files) == len(typewell_test_files), (
    "The number of horizontal and typewell test files does not match."
)

print(
    f"Successfully discovered "
    f"{len(horizontal_test_files)} Kaggle test well pairs."
)

Successfully discovered 3 Kaggle test well pairs.


#### 4.1 Load the Sample Submission

In [96]:
submission_candidates = [
    path
    for path in test_csv_files
    if "submission" in path.name.lower()
]

print("Submission candidates:")

for path in submission_candidates:
    print(path)

Submission candidates:
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\data\raw\test\sample_submission.csv


In [97]:
assert len(submission_candidates) >= 1, (
    "The sample submission file could not be found."
)

sample_submission_path = submission_candidates[0]

sample_submission_df = pd.read_csv(
    sample_submission_path
)

print("Sample submission path:")
print(sample_submission_path)

print("\nShape:")
print(sample_submission_df.shape)

print("\nColumns:")
print(sample_submission_df.columns.tolist())

display(sample_submission_df.head())

Sample submission path:
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\data\raw\test\sample_submission.csv

Shape:
(14151, 2)

Columns:
['id', 'tvt']


,id,tvt
0,000d7d20_1442,0.0
1,000d7d20_1443,0.0
2,000d7d20_1444,0.0
3,000d7d20_1445,0.0
4,000d7d20_1446,0.0


In [98]:
assert sample_submission_df.shape[0] > 0, (
    "The sample submission is empty."
)

assert sample_submission_df.shape[1] == 2, (
    "The sample submission should contain two columns."
)

print("Sample submission loaded successfully.")

Sample submission loaded successfully.


#### 4.2 Inspect One Test Well Pair

In [99]:
sample_horizontal_test_df = pd.read_csv(
    horizontal_test_files[0]
)

sample_typewell_test_df = pd.read_csv(
    typewell_test_files[0]
)

print("Horizontal test shape:")
print(sample_horizontal_test_df.shape)

print("\nHorizontal test columns:")
print(sample_horizontal_test_df.columns.tolist())

print("\nTypewell test shape:")
print(sample_typewell_test_df.shape)

print("\nTypewell test columns:")
print(sample_typewell_test_df.columns.tolist())

Horizontal test shape:
(5278, 6)

Horizontal test columns:
['MD', 'X', 'Y', 'Z', 'GR', 'TVT_input']

Typewell test shape:
(1296, 2)

Typewell test columns:
['TVT', 'GR']


In [100]:

display(sample_horizontal_test_df.head())
display(sample_typewell_test_df.head())

,MD,X,Y,Z,GR,TVT_input
0,11467.0,2983525.16,1069022.09,-9258.57,115.692586,11236.02
1,11468.0,2983525.18,1069022.30,-9259.55,115.584293,11237.05
2,11469.0,2983525.20,1069022.52,-9260.52,135.446960,11238.09
3,11470.0,2983525.22,1069022.73,-9261.50,140.401346,11239.12
4,11471.0,2983525.25,1069022.95,-9262.47,111.270638,11240.15


,TVT,GR
0,11223.95,126.11
1,11224.45,128.22
2,11224.95,128.72
3,11225.45,128.12
4,11225.95,125.29


In [101]:
required_horizontal_test_columns = {
    "MD",
    "X",
    "Y",
    "Z",
    "GR",
    "TVT_input",
}

required_typewell_test_columns = {
    "TVT",
    "GR",
}

missing_horizontal_columns = (
    required_horizontal_test_columns
    - set(sample_horizontal_test_df.columns)
)

missing_typewell_columns = (
    required_typewell_test_columns
    - set(sample_typewell_test_df.columns)
)

assert not missing_horizontal_columns, (
    f"Horizontal test columns missing: "
    f"{sorted(missing_horizontal_columns)}"
)

assert not missing_typewell_columns, (
    f"Typewell test columns missing: "
    f"{sorted(missing_typewell_columns)}"
)

print("Test dataset structure verified successfully.")

Test dataset structure verified successfully.


In [102]:
print(
    "Horizontal test rows:",
    len(sample_horizontal_test_df),
)

print(
    "Typewell reference rows:",
    len(sample_typewell_test_df),
)

print(
    "Missing TVT_input values:",
    sample_horizontal_test_df["TVT_input"].isna().sum(),
)

print(
    "Missing horizontal GR values:",
    sample_horizontal_test_df["GR"].isna().sum(),
)

Horizontal test rows: 5278
Typewell reference rows: 1296
Missing TVT_input values: 3836
Missing horizontal GR values: 2258


### 5. Load Final Selection from Notebook 07B

In [103]:
print("Notebook 07B artifacts:")

for path in sorted(NOTEBOOK_07B_RESULTS_DIR.iterdir()):
    print(path.name)

Notebook 07B artifacts:
figures
metadata
tables


In [104]:
print("Notebook 07B metadata files:")

for path in sorted(
    (NOTEBOOK_07B_RESULTS_DIR / "metadata").rglob("*")
):
    if path.is_file():
        print(path.relative_to(NOTEBOOK_07B_RESULTS_DIR))

Notebook 07B metadata files:
metadata\final_model_selection.json


In [105]:
print("\nNotebook 07B table files:")

for path in sorted(
    (NOTEBOOK_07B_RESULTS_DIR / "tables").rglob("*")
):
    if path.is_file():
        print(path.relative_to(NOTEBOOK_07B_RESULTS_DIR))


Notebook 07B table files:
tables\best_ensemble_oof_predictions.parquet
tables\best_model_per_well.csv
tables\computational_efficiency_summary.csv
tables\difficult_well_model_summary.csv
tables\difficult_wells.csv
tables\ensemble_weight_search.csv
tables\final_candidate_comparison.csv
tables\largest_prediction_errors.csv
tables\model_wins_summary.csv
tables\pairwise_statistical_tests.csv
tables\per_well_metrics.csv
tables\per_well_performance_summary.csv
tables\residual_statistics.csv


In [106]:
FINAL_CANDIDATE_PATH = (
    NOTEBOOK_07B_RESULTS_DIR
    / "tables"
    / "final_candidate_comparison.csv"
)

assert FINAL_CANDIDATE_PATH.exists(), (
    f"Final candidate comparison not found:\n"
    f"{FINAL_CANDIDATE_PATH}"
)

final_candidate_df = pd.read_csv(
    FINAL_CANDIDATE_PATH
)

display(final_candidate_df)

,candidate,candidate_type,oof_rmse,oof_mae
0,Extra Trees Optimized (0.80) + XGBoost Optimiz...,Weighted Ensemble,82.979455,38.103616
1,Extra Trees Optimized,Individual Model,83.586037,37.749300


In [107]:
ensemble_rows = final_candidate_df[
    final_candidate_df["candidate_type"]
    .str.strip()
    .str.lower()
    == "weighted ensemble"
].copy()

assert len(ensemble_rows) > 0, (
    "No weighted ensemble candidate was found."
)

selected_configuration = (
    ensemble_rows
    .sort_values("oof_rmse")
    .iloc[0]
)

display(selected_configuration.to_frame().T)

,candidate,candidate_type,oof_rmse,oof_mae
0,Extra Trees Optimized (0.80) + XGBoost Optimiz...,Weighted Ensemble,82.979455,38.103616


In [108]:
EXTRA_TREES_WEIGHT = 0.80
XGBOOST_WEIGHT = 0.20

assert np.isclose(
    EXTRA_TREES_WEIGHT + XGBOOST_WEIGHT,
    1.0,
)

print("Selected candidate:", selected_configuration["candidate"])
print("OOF RMSE:", selected_configuration["oof_rmse"])
print("OOF MAE:", selected_configuration["oof_mae"])
print("Extra Trees weight:", EXTRA_TREES_WEIGHT)
print("XGBoost weight:", XGBOOST_WEIGHT)

Selected candidate: Extra Trees Optimized (0.80) + XGBoost Optimized (0.20)
OOF RMSE: 82.97945521511976
OOF MAE: 38.103615891869296
Extra Trees weight: 0.8
XGBoost weight: 0.2


 #### 6. Inspect Project Utilities

In [109]:
from src.data_loader import (
    discover_training_wells,
    load_training_pair,
)

from src.feature_engineering import (
    create_horizontal_features,
)

import src.models as project_models

print("Project utilities imported successfully.")

Project utilities imported successfully.


### 7. Construct Full Training Dataset

This section loads every training well pair and applies the same reusable feature-engineering function used in the earlier modeling notebooks.

All available training wells are included because Notebook 08 performs final training without a validation split.

In [110]:
training_well_ids = discover_training_wells(
    train_dir=TRAIN_DIR,
)

print("Training wells discovered:", len(training_well_ids))
print("First five well IDs:", training_well_ids[:5])

assert len(training_well_ids) > 0, (
    "No training wells were discovered."
)

Training wells discovered: 773
First five well IDs: ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']


In [111]:
sample_training_well_id = training_well_ids[0]

typewell_sample_df, horizontal_sample_df = load_training_pair(
    well_id=sample_training_well_id,
    train_dir=TRAIN_DIR,
)

print("Sample well ID:", sample_training_well_id)

print("\nTypewell columns:")
print(typewell_sample_df.columns.tolist())

print("\nHorizontal columns:")
print(horizontal_sample_df.columns.tolist())

print("\nTypewell shape:", typewell_sample_df.shape)
print("Horizontal shape:", horizontal_sample_df.shape)

display(typewell_sample_df.head())
display(horizontal_sample_df.head())

Sample well ID: 000d7d20

Typewell columns:
['TVT', 'GR', 'Geology']

Horizontal columns:
['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input']

Typewell shape: (1296, 3)
Horizontal shape: (5278, 13)


,TVT,GR,Geology
0,11223.95,126.11,NaN
1,11224.45,128.22,NaN
2,11224.95,128.72,NaN
3,11225.45,128.12,NaN
4,11225.95,125.29,NaN


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15


In [112]:
required_training_horizontal_columns = {
    "MD",
    "X",
    "Y",
    "Z",
    "GR",
    "TVT_input",
    "TVT",
}

missing_training_columns = (
    required_training_horizontal_columns
    - set(horizontal_sample_df.columns)
)

assert not missing_training_columns, (
    f"Training horizontal data is missing columns: "
    f"{sorted(missing_training_columns)}"
)

print("Sample training well structure verified.")

Sample training well structure verified.


In [113]:
sample_features_df = create_horizontal_features(
    horizontal_df=horizontal_sample_df,
    well_id=sample_training_well_id,
)

print("Sample feature shape:", sample_features_df.shape)
print("Sample feature columns:", sample_features_df.columns.tolist())

display(sample_features_df.head())

assert len(sample_features_df) == len(horizontal_sample_df), (
    "Feature row count does not match the horizontal dataset."
)

print("Sample feature construction succeeded.")

Sample feature shape: (5278, 26)
Sample feature columns: ['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input', 'well_id', 'MD_relative', 'X_relative', 'Y_relative', 'Z_relative', 'MD_diff', 'X_diff', 'Y_diff', 'Z_diff', 'GR_diff', 'TVT_input_diff', 'horizontal_distance', 'spatial_distance']


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input,well_id,MD_relative,X_relative,Y_relative,Z_relative,MD_diff,X_diff,Y_diff,Z_diff,GR_diff,TVT_input_diff,horizontal_distance,spatial_distance
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02,000d7d20,0.0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05,000d7d20,1.0,0.02,0.21,-0.98,1.0,0.02,0.21,-0.98,-0.108293,1.03,0.210950,1.002447
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09,000d7d20,2.0,0.04,0.43,-1.95,1.0,0.02,0.22,-0.97,19.862666,1.04,0.431856,1.997248
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12,000d7d20,3.0,0.06,0.64,-2.93,1.0,0.02,0.21,-0.98,4.954386,1.03,0.642806,2.999683
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15,000d7d20,4.0,0.09,0.86,-3.90,1.0,0.03,0.22,-0.97,-29.130707,1.03,0.864696,3.994709


Sample feature construction succeeded.


In [114]:
training_feature_frames = []
failed_training_wells = []

training_construction_start = time.time()

for well_index, well_id in enumerate(
    training_well_ids,
    start=1,
):
    try:
        # Important:
        # load_training_pair returns typewell first,
        # horizontal well second.
        typewell_df, horizontal_df = load_training_pair(
            well_id=well_id,
            train_dir=TRAIN_DIR,
        )

        if "TVT" not in horizontal_df.columns:
            raise ValueError(
                f"Horizontal training well {well_id} "
                f"does not contain the TVT target."
            )

        target_tvt = horizontal_df["TVT"].copy()

        well_features_df = create_horizontal_features(
            horizontal_df=horizontal_df,
            well_id=well_id,
        )

        if len(well_features_df) != len(target_tvt):
            raise ValueError(
                f"Feature-target row mismatch for well {well_id}: "
                f"{len(well_features_df)} features versus "
                f"{len(target_tvt)} targets."
            )

        well_features_df = well_features_df.copy()

        well_features_df["target_tvt"] = (
            target_tvt.to_numpy()
        )

        well_features_df["well_id"] = well_id

        training_feature_frames.append(
            well_features_df
        )

    except Exception as error:
        failed_training_wells.append({
            "well_id": well_id,
            "error_type": type(error).__name__,
            "error": str(error),
        })

    if (
        well_index == 1
        or well_index % 50 == 0
        or well_index == len(training_well_ids)
    ):
        print(
            f"Processed {well_index:,} / "
            f"{len(training_well_ids):,} wells | "
            f"Successful: {len(training_feature_frames):,} | "
            f"Failed: {len(failed_training_wells):,}"
        )

training_construction_seconds = (
    time.time() - training_construction_start
)

print("\nTraining-well processing complete.")
print(
    "Construction time:",
    round(training_construction_seconds, 2),
    "seconds",
)
print("Successful wells:", len(training_feature_frames))
print("Failed wells:", len(failed_training_wells))

Processed 1 / 773 wells | Successful: 1 | Failed: 0
Processed 50 / 773 wells | Successful: 50 | Failed: 0
Processed 100 / 773 wells | Successful: 100 | Failed: 0
Processed 150 / 773 wells | Successful: 150 | Failed: 0
Processed 200 / 773 wells | Successful: 200 | Failed: 0
Processed 250 / 773 wells | Successful: 250 | Failed: 0
Processed 300 / 773 wells | Successful: 300 | Failed: 0
Processed 350 / 773 wells | Successful: 350 | Failed: 0
Processed 400 / 773 wells | Successful: 400 | Failed: 0
Processed 450 / 773 wells | Successful: 450 | Failed: 0
Processed 500 / 773 wells | Successful: 500 | Failed: 0
Processed 550 / 773 wells | Successful: 550 | Failed: 0
Processed 600 / 773 wells | Successful: 600 | Failed: 0
Processed 650 / 773 wells | Successful: 650 | Failed: 0
Processed 700 / 773 wells | Successful: 700 | Failed: 0
Processed 750 / 773 wells | Successful: 750 | Failed: 0
Processed 773 / 773 wells | Successful: 773 | Failed: 0

Training-well processing complete.
Construction time:

In [115]:
failed_training_wells_df = pd.DataFrame(
    failed_training_wells
)

if failed_training_wells_df.empty:
    print("All training wells were processed successfully.")

else:
    display(failed_training_wells_df)

    failed_training_wells_path = (
        NOTEBOOK_08_RESULTS_DIR
        / "failed_training_wells.csv"
    )

    failed_training_wells_df.to_csv(
        failed_training_wells_path,
        index=False,
    )

    raise RuntimeError(
        f"{len(failed_training_wells_df)} training wells failed. "
        f"Review: {failed_training_wells_path}"
    )

All training wells were processed successfully.


In [116]:
assert len(training_feature_frames) > 0, (
    "No successfully processed training wells are available."
)

full_training_df = pd.concat(
    training_feature_frames,
    ignore_index=True,
)

print("Full training dataset constructed.")
print("Shape:", full_training_df.shape)
print(
    "Unique wells:",
    full_training_df["well_id"].nunique(),
)

display(full_training_df.head())

Full training dataset constructed.
Shape: (5092255, 27)
Unique wells: 773


,MD,X,Y,Z,ANCC,ASTNU,ASTNL,EGFDU,EGFDL,BUDA,TVT,GR,TVT_input,well_id,MD_relative,X_relative,Y_relative,Z_relative,MD_diff,X_diff,Y_diff,Z_diff,GR_diff,TVT_input_diff,horizontal_distance,spatial_distance,target_tvt
0,11467.0,2983525.16,1069022.09,-9258.57,-9395.81,-9569.86,-9597.64,-9670.99,-9705.96,-9846.35,11236.02,115.692586,11236.02,000d7d20,0.0,0.00,0.00,0.00,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,11236.02
1,11468.0,2983525.18,1069022.30,-9259.55,-9395.75,-9569.80,-9597.58,-9670.93,-9705.90,-9846.29,11237.05,115.584293,11237.05,000d7d20,1.0,0.02,0.21,-0.98,1.0,0.02,0.21,-0.98,-0.108293,1.03,0.210950,1.002447,11237.05
2,11469.0,2983525.20,1069022.52,-9260.52,-9395.69,-9569.74,-9597.52,-9670.87,-9705.84,-9846.23,11238.09,135.446960,11238.09,000d7d20,2.0,0.04,0.43,-1.95,1.0,0.02,0.22,-0.97,19.862666,1.04,0.431856,1.997248,11238.09
3,11470.0,2983525.22,1069022.73,-9261.50,-9395.64,-9569.69,-9597.47,-9670.82,-9705.79,-9846.18,11239.12,140.401346,11239.12,000d7d20,3.0,0.06,0.64,-2.93,1.0,0.02,0.21,-0.98,4.954386,1.03,0.642806,2.999683,11239.12
4,11471.0,2983525.25,1069022.95,-9262.47,-9395.58,-9569.63,-9597.41,-9670.76,-9705.73,-9846.12,11240.15,111.270638,11240.15,000d7d20,4.0,0.09,0.86,-3.90,1.0,0.03,0.22,-0.97,-29.130707,1.03,0.864696,3.994709,11240.15


In [117]:
NON_FEATURE_COLUMNS = [
    "well_id",
    "target_tvt",
]

FEATURE_COLUMNS = [
    column
    for column in full_training_df.columns
    if column not in NON_FEATURE_COLUMNS
]

X_full = full_training_df[
    FEATURE_COLUMNS
].copy()

y_full = full_training_df[
    "target_tvt"
].copy()

training_groups = full_training_df[
    "well_id"
].copy()

print("Feature matrix shape:", X_full.shape)
print("Target shape:", y_full.shape)
print("Groups shape:", training_groups.shape)
print("Number of features:", len(FEATURE_COLUMNS))

print("\nFeature columns:")

for column in FEATURE_COLUMNS:
    print("-", column)

Feature matrix shape: (5092255, 25)
Target shape: (5092255,)
Groups shape: (5092255,)
Number of features: 25

Feature columns:
- MD
- X
- Y
- Z
- ANCC
- ASTNU
- ASTNL
- EGFDU
- EGFDL
- BUDA
- TVT
- GR
- TVT_input
- MD_relative
- X_relative
- Y_relative
- Z_relative
- MD_diff
- X_diff
- Y_diff
- Z_diff
- GR_diff
- TVT_input_diff
- horizontal_distance
- spatial_distance


In [118]:
assert len(X_full) == len(y_full), (
    "Feature and target row counts do not match."
)

assert len(X_full) == len(training_groups), (
    "Feature and group row counts do not match."
)

assert full_training_df["well_id"].nunique() == len(
    training_well_ids
), (
    "Not all discovered training wells are represented."
)

assert not y_full.isna().any(), (
    "The training target contains missing values."
)

assert not X_full.columns.duplicated().any(), (
    "Duplicate feature columns were found."
)

non_numeric_feature_columns = (
    X_full
    .select_dtypes(exclude=[np.number])
    .columns
    .tolist()
)

assert not non_numeric_feature_columns, (
    f"Non-numeric model features found: "
    f"{non_numeric_feature_columns}"
)

infinite_value_count = int(
    np.isinf(
        X_full.to_numpy(dtype=float)
    ).sum()
)

print("Missing feature values:", int(X_full.isna().sum().sum()))
print("Infinite feature values:", infinite_value_count)
print("Missing target values:", int(y_full.isna().sum()))
print("Unique training wells:", training_groups.nunique())

assert infinite_value_count == 0, (
    "The training feature matrix contains infinite values."
)

print("\nFull training dataset validated successfully.")

Missing feature values: 11456203
Infinite feature values: 0
Missing target values: 0
Unique training wells: 773

Full training dataset validated successfully.


In [119]:
feature_columns_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "final_feature_columns.json"
)

with open(
    feature_columns_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        FEATURE_COLUMNS,
        file,
        indent=4,
    )

training_dataset_summary_df = pd.DataFrame([
    {
        "number_of_wells": training_groups.nunique(),
        "number_of_rows": len(X_full),
        "number_of_features": len(FEATURE_COLUMNS),
        "missing_feature_values": int(
            X_full.isna().sum().sum()
        ),
        "missing_target_values": int(
            y_full.isna().sum()
        ),
        "construction_seconds": round(
            training_construction_seconds,
            4,
        ),
    }
])

training_summary_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "training_dataset_summary.csv"
)

training_dataset_summary_df.to_csv(
    training_summary_path,
    index=False,
)

display(training_dataset_summary_df)

print("Feature columns saved to:", feature_columns_path)
print("Training summary saved to:", training_summary_path)

,number_of_wells,number_of_rows,number_of_features,missing_feature_values,missing_target_values,construction_seconds
0,773,5092255,25,11456203,0,34.9925


Feature columns saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\final_feature_columns.json
Training summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\training_dataset_summary.csv


In [120]:
missing_feature_summary = (
    X_full.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

missing_feature_summary["missing_percentage"] = (
    missing_feature_summary["missing_count"]
    / len(X_full)
    * 100
)

display(
    missing_feature_summary[
        missing_feature_summary["missing_count"] > 0
    ]
)

,missing_count,missing_percentage
TVT_input_diff,3784762,74.323890
TVT_input,3783989,74.308710
GR_diff,2324687,45.651426
GR,1507972,29.613050
ANCC,45634,0.896145
EGFDL,6067,0.119142
Y_diff,773,0.015180
MD_diff,773,0.015180
X_diff,773,0.015180
Z_diff,773,0.015180


In [121]:
missing_feature_summary_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "training_missing_feature_summary.csv"
)

missing_feature_summary.reset_index(
    names="feature"
).to_csv(
    missing_feature_summary_path,
    index=False,
)

print(
    "Missing-feature summary saved to:",
    missing_feature_summary_path,
)

Missing-feature summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\training_missing_feature_summary.csv


### 8. Train Final Production Models

This section trains the two models selected for the final weighted ensemble using the complete training dataset.

No validation split is used because model selection and evaluation were completed in Notebook 07.

The final production models are:

- Optimized Extra Trees Regressor
- Optimized XGBoost Regressor

Both trained pipelines are saved immediately after training so they can be reused for Kaggle test prediction and future inference.

#### 8.1 Define Final Model Features

In [156]:

# Use only features available in both train and Kaggle test
# horizontals. Formation markers (ANCC, ASTNU, ASTNL, EGFDU,
# EGFDL, BUDA) exist in training files but not in the test set.
MODEL_FEATURE_COLUMNS = [
    "MD",
    "GR",
    "TVT_input",
    "X",
    "Y",
    "Z",
]

missing_model_features = [
    column
    for column in MODEL_FEATURE_COLUMNS
    if column not in X_full.columns
]

assert not missing_model_features, (
    "The training dataset is missing required model features: "
    f"{missing_model_features}"
)

X_final = X_full[
    MODEL_FEATURE_COLUMNS
].copy()

print("Final feature matrix shape:", X_final.shape)
print("Final target shape:", y_full.shape)
print("Number of model features:", len(MODEL_FEATURE_COLUMNS))

print("\nFinal model features:")

for column in MODEL_FEATURE_COLUMNS:
    print("-", column)

Final feature matrix shape: (5092255, 6)
Final target shape: (5092255,)
Number of model features: 6

Final model features:
- MD
- GR
- TVT_input
- X
- Y
- Z


In [157]:


assert len(X_final) == len(y_full), (
    "Final feature and target row counts do not match."
)

assert not y_full.isna().any(), (
    "The final training target contains missing values."
)

assert not X_final.columns.duplicated().any(), (
    "Duplicate final feature columns were found."
)

final_missing_values = int(
    X_final.isna().sum().sum()
)

final_infinite_values = int(
    np.isinf(
        X_final.to_numpy(dtype=float)
    ).sum()
)

print("Missing feature values:", final_missing_values)
print("Infinite feature values:", final_infinite_values)
print("Missing target values:", int(y_full.isna().sum()))

assert final_infinite_values == 0, (
    "The final feature matrix contains infinite values."
)

print("Final training data validated successfully.")

Missing feature values: 5291961
Infinite feature values: 0
Missing target values: 0
Final training data validated successfully.


#### 8.2 Save the final feature list

In [158]:
final_model_features_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "final_model_feature_columns.json"
)

with open(
    final_model_features_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        MODEL_FEATURE_COLUMNS,
        file,
        indent=4,
    )

print(
    "Final model features saved to:",
    final_model_features_path,
)

Final model features saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\final_model_feature_columns.json


#### 8.3 Prepare Optimized Extra Trees Pipeline

In [159]:
NOTEBOOK_06_MODELS_DIR = (
    MODELS_DIR
    / "notebook_06"
)

OPTIMIZED_EXTRA_TREES_PATH = (
    NOTEBOOK_06_MODELS_DIR
    / "best_optimized_pipeline.joblib"
)

assert OPTIMIZED_EXTRA_TREES_PATH.exists(), (
    "The optimized Extra Trees pipeline was not found:\n"
    f"{OPTIMIZED_EXTRA_TREES_PATH}"
)

# Reuse optimized hyperparameters from notebook 06, but rebuild
# the preprocessor for MODEL_FEATURE_COLUMNS. The saved pipeline
# still selects formation-marker columns that are absent from
# the Kaggle test set.
saved_extra_trees_pipeline = joblib.load(
    OPTIMIZED_EXTRA_TREES_PATH
)

optimized_extra_trees_params = (
    saved_extra_trees_pipeline
    .named_steps["model"]
    .get_params()
)

final_extra_trees_pipeline = (
    project_models.create_extra_trees_pipeline(
        feature_columns=MODEL_FEATURE_COLUMNS,
        model_params={
            "n_estimators": optimized_extra_trees_params[
                "n_estimators"
            ],
            "max_depth": optimized_extra_trees_params[
                "max_depth"
            ],
            "min_samples_leaf": optimized_extra_trees_params[
                "min_samples_leaf"
            ],
            "bootstrap": optimized_extra_trees_params[
                "bootstrap"
            ],
            "max_samples": optimized_extra_trees_params[
                "max_samples"
            ],
            "max_features": optimized_extra_trees_params[
                "max_features"
            ],
            "random_state": optimized_extra_trees_params[
                "random_state"
            ],
            "n_jobs": optimized_extra_trees_params[
                "n_jobs"
            ],
        },
    )
)

print("Optimized Extra Trees pipeline prepared.")
print(
    "Feature columns:",
    MODEL_FEATURE_COLUMNS,
)

print(
    "Estimator:",
    type(
        final_extra_trees_pipeline.named_steps["model"]
    ).__name__,
)

print(
    "Number of estimators:",
    final_extra_trees_pipeline
    .named_steps["model"]
    .get_params()["n_estimators"],
)


Optimized Extra Trees pipeline prepared.
Feature columns: ['MD', 'GR', 'TVT_input', 'X', 'Y', 'Z']
Estimator: ExtraTreesRegressor
Number of estimators: 150


#### 8.4 Train the final Extra Trees model

In [160]:
extra_trees_training_start = time.time()

final_extra_trees_pipeline.fit(
    X_final,
    y_full,
)

extra_trees_training_seconds = (
    time.time()
    - extra_trees_training_start
)

print("Final Extra Trees training completed.")

print(
    "Training time:",
    round(extra_trees_training_seconds, 2),
    "seconds",
)

print(
    "Training time:",
    round(extra_trees_training_seconds / 60, 2),
    "minutes",
)

Final Extra Trees training completed.
Training time: 99.25 seconds
Training time: 1.65 minutes


In [161]:

FINAL_EXTRA_TREES_MODEL_PATH = (
    NOTEBOOK_08_MODELS_DIR
    / "final_extra_trees_pipeline.joblib"
)

joblib.dump(
    final_extra_trees_pipeline,
    FINAL_EXTRA_TREES_MODEL_PATH,
)

print(
    "Final Extra Trees pipeline saved to:",
    FINAL_EXTRA_TREES_MODEL_PATH,
)

print(
    "Model file size:",
    round(
        FINAL_EXTRA_TREES_MODEL_PATH.stat().st_size
        / (1024 ** 2),
        2,
    ),
    "MB",
)

Final Extra Trees pipeline saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\models\notebook_08\final_extra_trees_pipeline.joblib
Model file size: 290.09 MB


#### 8.5 Prepare the optimized XGBoost pipeline

In [162]:
final_xgboost_pipeline = (
    project_models.create_xgboost_pipeline(
        feature_columns=MODEL_FEATURE_COLUMNS
    )
)

final_xgboost_pipeline.set_params(
    model__n_estimators=500,
)

xgboost_model_params = (
    final_xgboost_pipeline
    .named_steps["model"]
    .get_params()
)

print("Optimized XGBoost pipeline prepared.")

print(
    "Estimator:",
    type(
        final_xgboost_pipeline.named_steps["model"]
    ).__name__,
)

print(
    "Number of estimators:",
    xgboost_model_params["n_estimators"],
)

print(
    "Maximum depth:",
    xgboost_model_params["max_depth"],
)

print(
    "Learning rate:",
    xgboost_model_params["learning_rate"],
)

Optimized XGBoost pipeline prepared.
Estimator: XGBRegressor
Number of estimators: 500
Maximum depth: 8
Learning rate: 0.05


In [163]:
xgboost_configuration_df = pd.DataFrame([
    {
        "parameter": "n_estimators",
        "value": xgboost_model_params.get(
            "n_estimators"
        ),
    },
    {
        "parameter": "learning_rate",
        "value": xgboost_model_params.get(
            "learning_rate"
        ),
    },
    {
        "parameter": "max_depth",
        "value": xgboost_model_params.get(
            "max_depth"
        ),
    },
    {
        "parameter": "min_child_weight",
        "value": xgboost_model_params.get(
            "min_child_weight"
        ),
    },
    {
        "parameter": "subsample",
        "value": xgboost_model_params.get(
            "subsample"
        ),
    },
    {
        "parameter": "colsample_bytree",
        "value": xgboost_model_params.get(
            "colsample_bytree"
        ),
    },
    {
        "parameter": "reg_alpha",
        "value": xgboost_model_params.get(
            "reg_alpha"
        ),
    },
    {
        "parameter": "reg_lambda",
        "value": xgboost_model_params.get(
            "reg_lambda"
        ),
    },
    {
        "parameter": "n_jobs",
        "value": xgboost_model_params.get(
            "n_jobs"
        ),
    },
    {
        "parameter": "random_state",
        "value": xgboost_model_params.get(
            "random_state"
        ),
    },
])

display(xgboost_configuration_df)

,parameter,value
0,n_estimators,500.00
1,learning_rate,0.05
2,max_depth,8.00
3,min_child_weight,20.00
4,subsample,0.50
5,colsample_bytree,0.80
6,reg_alpha,0.10
7,reg_lambda,1.00
8,n_jobs,4.00
9,random_state,42.00


#### 8.6 Train the final XGBoost model

In [164]:
xgboost_training_start = time.time()

final_xgboost_pipeline.fit(
    X_final,
    y_full,
)

xgboost_training_seconds = (
    time.time()
    - xgboost_training_start
)

print("Final XGBoost training completed.")

print(
    "Training time:",
    round(xgboost_training_seconds, 2),
    "seconds",
)

print(
    "Training time:",
    round(xgboost_training_seconds / 60, 2),
    "minutes",
)

Final XGBoost training completed.
Training time: 197.37 seconds
Training time: 3.29 minutes


In [165]:
FINAL_XGBOOST_MODEL_PATH = (
    NOTEBOOK_08_MODELS_DIR
    / "final_xgboost_pipeline.joblib"
)

joblib.dump(
    final_xgboost_pipeline,
    FINAL_XGBOOST_MODEL_PATH,
)

print(
    "Final XGBoost pipeline saved to:",
    FINAL_XGBOOST_MODEL_PATH,
)

print(
    "Model file size:",
    round(
        FINAL_XGBOOST_MODEL_PATH.stat().st_size
        / (1024 ** 2),
        2,
    ),
    "MB",
)

Final XGBoost pipeline saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\models\notebook_08\final_xgboost_pipeline.joblib
Model file size: 7.6 MB


#### 8.7 Verify the saved models

In [166]:
final_model_artifacts = {
    "Extra Trees Optimized": (
        FINAL_EXTRA_TREES_MODEL_PATH
    ),
    "XGBoost Optimized": (
        FINAL_XGBOOST_MODEL_PATH
    ),
}

model_verification_rows = []

for model_name, model_path in (
    final_model_artifacts.items()
):
    model_verification_rows.append({
        "model": model_name,
        "path": str(model_path),
        "exists": model_path.exists(),
        "size_mb": (
            round(
                model_path.stat().st_size
                / (1024 ** 2),
                4,
            )
            if model_path.exists()
            else np.nan
        ),
    })

final_model_verification_df = pd.DataFrame(
    model_verification_rows
)

display(final_model_verification_df)

assert final_model_verification_df[
    "exists"
].all(), (
    "One or more final model artifacts were not saved."
)

print("Both final model artifacts verified successfully.")

,model,path,exists,size_mb
0,Extra Trees Optimized,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,290.0853
1,XGBoost Optimized,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,7.5987


Both final model artifacts verified successfully.


#### 8.8 Save the training summary

In [167]:
final_training_summary_df = pd.DataFrame([
    {
        "model": "Extra Trees Optimized",
        "number_of_rows": len(X_final),
        "number_of_wells": training_groups.nunique(),
        "number_of_features": len(
            MODEL_FEATURE_COLUMNS
        ),
        "training_seconds": round(
            extra_trees_training_seconds,
            4,
        ),
        "training_minutes": round(
            extra_trees_training_seconds / 60,
            4,
        ),
        "ensemble_weight": EXTRA_TREES_WEIGHT,
        "model_path": str(
            FINAL_EXTRA_TREES_MODEL_PATH
        ),
    },
    {
        "model": "XGBoost Optimized",
        "number_of_rows": len(X_final),
        "number_of_wells": training_groups.nunique(),
        "number_of_features": len(
            MODEL_FEATURE_COLUMNS
        ),
        "training_seconds": round(
            xgboost_training_seconds,
            4,
        ),
        "training_minutes": round(
            xgboost_training_seconds / 60,
            4,
        ),
        "ensemble_weight": XGBOOST_WEIGHT,
        "model_path": str(
            FINAL_XGBOOST_MODEL_PATH
        ),
    },
])

final_training_summary_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "final_model_training_summary.csv"
)

final_training_summary_df.to_csv(
    final_training_summary_path,
    index=False,
)

display(final_training_summary_df)

print(
    "Final training summary saved to:",
    final_training_summary_path,
)

,model,number_of_rows,number_of_wells,number_of_features,training_seconds,training_minutes,ensemble_weight,model_path
0,Extra Trees Optimized,5092255,773,6,99.2522,1.6542,0.8,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...
1,XGBoost Optimized,5092255,773,6,197.3748,3.2896,0.2,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...


Final training summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\final_model_training_summary.csv


### 9. Construct Kaggle Test Features

This section prepares the Kaggle test dataset for final model inference.

The test horizontal wells are processed using the same feature-engineering utilities used for the training data. Original file order and row order are preserved so predictions can be aligned exactly with the Kaggle sample submission.

The final output contains the 12 validated features required by the optimized Extra Trees and XGBoost pipelines.

#### 9.1 Discover and pair the test files

In [168]:


test_horizontal_files = sorted(
    TEST_DIR.glob("*__horizontal_well.csv")
)

test_typewell_files = sorted(
    TEST_DIR.glob("*__typewell.csv")
)

assert test_horizontal_files, (
    f"No horizontal test files were found in: {TEST_DIR}"
)

assert test_typewell_files, (
    f"No typewell test files were found in: {TEST_DIR}"
)


def extract_well_id(file_path):
    """
    Extract the well identifier from a paired test filename.

    Example:
    000d7d20__horizontal_well.csv -> 000d7d20
    """
    return file_path.name.split("__")[0]


horizontal_file_map = {
    extract_well_id(file_path): file_path
    for file_path in test_horizontal_files
}

typewell_file_map = {
    extract_well_id(file_path): file_path
    for file_path in test_typewell_files
}

horizontal_well_ids = set(
    horizontal_file_map.keys()
)

typewell_well_ids = set(
    typewell_file_map.keys()
)

missing_typewell_ids = sorted(
    horizontal_well_ids
    - typewell_well_ids
)

missing_horizontal_ids = sorted(
    typewell_well_ids
    - horizontal_well_ids
)

assert not missing_typewell_ids, (
    "Some horizontal wells do not have matching typewell files: "
    f"{missing_typewell_ids}"
)

assert not missing_horizontal_ids, (
    "Some typewell files do not have matching horizontal wells: "
    f"{missing_horizontal_ids}"
)

test_well_ids = sorted(
    horizontal_well_ids
)

print("Number of test well pairs:", len(test_well_ids))
print("Test well IDs:", test_well_ids)

Number of test well pairs: 3
Test well IDs: ['000d7d20', '00bbac68', '00e12e8b']


#### 9.2 Inspect test schemas

In [169]:


test_schema_rows = []

for well_id in test_well_ids:

    horizontal_df = pd.read_csv(
        horizontal_file_map[well_id]
    )

    typewell_df = pd.read_csv(
        typewell_file_map[well_id]
    )

    test_schema_rows.append({
        "well_id": well_id,
        "horizontal_rows": len(horizontal_df),
        "typewell_rows": len(typewell_df),
        "missing_tvt_input_rows": int(
            horizontal_df["TVT_input"].isna().sum()
        ),
        "available_horizontal_columns": ", ".join(
            horizontal_df.columns.tolist()
        ),
    })

test_schema_df = pd.DataFrame(
    test_schema_rows
)

display(test_schema_df)

print(
    "Total horizontal rows:",
    test_schema_df["horizontal_rows"].sum(),
)

print(
    "Total rows requiring predictions:",
    test_schema_df["missing_tvt_input_rows"].sum(),
)

print(
    "Sample submission rows:",
    len(sample_submission_df),
)

,well_id,horizontal_rows,typewell_rows,missing_tvt_input_rows,available_horizontal_columns
0,000d7d20,5278,1296,3836,"MD, X, Y, Z, GR, TVT_input"
1,00bbac68,7559,1946,6014,"MD, X, Y, Z, GR, TVT_input"
2,00e12e8b,6384,2556,4301,"MD, X, Y, Z, GR, TVT_input"


Total horizontal rows: 19221
Total rows requiring predictions: 14151
Sample submission rows: 14151


In [170]:
assert (
    test_schema_df[
        "missing_tvt_input_rows"
    ].sum()
    == len(sample_submission_df)
), (
    "The number of missing TVT_input rows does not "
    "match the sample submission row count."
)

print(
    "Prediction-row count matches the sample submission."
)

Prediction-row count matches the sample submission.


#### 9.3 Construct Test Features

In [171]:

test_feature_frames = []
test_construction_rows = []

test_feature_start = time.time()

global_prediction_position = 0

for well_order, well_id in enumerate(
    test_well_ids
):

    horizontal_path = horizontal_file_map[
        well_id
    ]

    typewell_path = typewell_file_map[
        well_id
    ]

    horizontal_df = pd.read_csv(
        horizontal_path
    )

    typewell_df = pd.read_csv(
        typewell_path
    )

    original_row_count = len(
        horizontal_df
    )

    # Kaggle requires predictions only for rows
    # where TVT_input is missing.
    prediction_mask = (
        horizontal_df["TVT_input"].isna()
    )

    number_of_prediction_rows = int(
        prediction_mask.sum()
    )

    # Apply feature engineering to the complete well
    # before selecting prediction rows.
    engineered_horizontal_df = (
        create_horizontal_features(
            horizontal_df.copy(),
            well_id=well_id,
        )
    )

    assert len(
        engineered_horizontal_df
    ) == original_row_count, (
        f"Feature engineering changed the row count "
        f"for test well {well_id}."
    )

    missing_columns = [
        column
        for column in MODEL_FEATURE_COLUMNS
        if column
        not in engineered_horizontal_df.columns
    ]

    assert not missing_columns, (
        f"Feature engineering for well {well_id} "
        f"did not create required columns: "
        f"{missing_columns}"
    )

    # Select only rows requiring predictions.
    well_prediction_features_df = (
        engineered_horizontal_df
        .loc[
            prediction_mask,
            MODEL_FEATURE_COLUMNS,
        ]
        .copy()
        .reset_index(drop=True)
    )

    assert len(
        well_prediction_features_df
    ) == number_of_prediction_rows, (
        f"Prediction-row count mismatch "
        f"for test well {well_id}."
    )

    original_prediction_positions = (
        np.flatnonzero(
            prediction_mask.to_numpy()
        )
    )

    well_prediction_features_df.insert(
        0,
        "original_row_position",
        original_prediction_positions,
    )

    well_prediction_features_df.insert(
        0,
        "well_order",
        well_order,
    )

    well_prediction_features_df.insert(
        0,
        "well_id",
        well_id,
    )

    well_prediction_features_df[
        "global_prediction_position"
    ] = np.arange(
        global_prediction_position,
        global_prediction_position
        + number_of_prediction_rows,
    )

    global_prediction_position += (
        number_of_prediction_rows
    )

    test_feature_frames.append(
        well_prediction_features_df
    )

    test_construction_rows.append({
        "well_id": well_id,
        "horizontal_file": (
            horizontal_path.name
        ),
        "typewell_file": (
            typewell_path.name
        ),
        "total_horizontal_rows": (
            original_row_count
        ),
        "prediction_rows": (
            number_of_prediction_rows
        ),
        "known_tvt_input_rows": int(
            horizontal_df[
                "TVT_input"
            ].notna().sum()
        ),
        "typewell_rows": len(
            typewell_df
        ),
        "missing_feature_values": int(
            well_prediction_features_df[
                MODEL_FEATURE_COLUMNS
            ]
            .isna()
            .sum()
            .sum()
        ),
    })

assert test_feature_frames, (
    "No test feature frames were constructed."
)

test_features_with_metadata_df = (
    pd.concat(
        test_feature_frames,
        ignore_index=True,
    )
)

test_feature_seconds = (
    time.time()
    - test_feature_start
)

test_construction_summary_df = (
    pd.DataFrame(
        test_construction_rows
    )
)

display(
    test_construction_summary_df
)

print(
    "Total constructed prediction rows:",
    len(test_features_with_metadata_df),
)

print(
    "Expected submission rows:",
    len(sample_submission_df),
)

print(
    "Construction time:",
    round(
        test_feature_seconds,
        2,
    ),
    "seconds",
)

assert len(
    test_features_with_metadata_df
) == len(
    sample_submission_df
), (
    "Constructed prediction rows do not match "
    "the sample submission.\n"
    f"Constructed: "
    f"{len(test_features_with_metadata_df):,}\n"
    f"Expected: "
    f"{len(sample_submission_df):,}"
)

print(
    "Test feature construction completed successfully."
)

,well_id,horizontal_file,typewell_file,total_horizontal_rows,prediction_rows,known_tvt_input_rows,typewell_rows,missing_feature_values
0,000d7d20,000d7d20__horizontal_well.csv,000d7d20__typewell.csv,5278,3836,1442,1296,5652
1,00bbac68,00bbac68__horizontal_well.csv,00bbac68__typewell.csv,7559,6014,1545,1946,6846
2,00e12e8b,00e12e8b__horizontal_well.csv,00e12e8b__typewell.csv,6384,4301,2083,2556,4719


Total constructed prediction rows: 14151
Expected submission rows: 14151
Construction time: 0.09 seconds
Test feature construction completed successfully.


#### 9.4 Create Final Test Inference Matrix

In [172]:

X_test_final = (
    test_features_with_metadata_df[
        MODEL_FEATURE_COLUMNS
    ]
    .copy()
)

test_metadata_df = (
    test_features_with_metadata_df[
        [
            "well_id",
            "well_order",
            "original_row_position",
            "global_prediction_position",
        ]
    ]
    .copy()
)

print(
    "Final test feature shape:",
    X_test_final.shape,
)

print(
    "Test metadata shape:",
    test_metadata_df.shape,
)

print(
    "Number of test wells:",
    test_metadata_df[
        "well_id"
    ].nunique(),
)

print(
    "Missing feature values:",
    int(
        X_test_final
        .isna()
        .sum()
        .sum()
    ),
)

Final test feature shape: (14151, 6)
Test metadata shape: (14151, 4)
Number of test wells: 3
Missing feature values: 17217


#### 9.5 Validate Final Test Matrix

In [173]:


assert len(X_test_final) == len(
    sample_submission_df
), (
    "Constructed prediction rows do not match "
    "the sample submission."
)

assert list(
    X_test_final.columns
) == MODEL_FEATURE_COLUMNS, (
    "Test feature columns are not in the expected order."
)

assert not (
    X_test_final.columns.duplicated()
).any(), (
    "Duplicate test feature columns were found."
)

test_infinite_values = int(
    np.isinf(
        X_test_final.to_numpy(
            dtype=float
        )
    ).sum()
)

assert test_infinite_values == 0, (
    "The final test feature matrix contains "
    "infinite values."
)

print(
    "Constructed prediction rows:",
    f"{len(X_test_final):,}",
)

print(
    "Sample submission rows:",
    f"{len(sample_submission_df):,}",
)

print(
    "Missing feature values:",
    int(
        X_test_final
        .isna()
        .sum()
        .sum()
    ),
)

print(
    "Infinite feature values:",
    test_infinite_values,
)

print(
    "Final test feature validation successful."
)

Constructed prediction rows: 14,151
Sample submission rows: 14,151
Missing feature values: 17217
Infinite feature values: 0
Final test feature validation successful.


#### 9.6 Attach Kaggle Submission IDs

In [174]:


assert "id" in sample_submission_df.columns, (
    "The sample submission does not contain "
    "an 'id' column."
)

test_metadata_df.insert(
    0,
    "id",
    sample_submission_df[
        "id"
    ].to_numpy(),
)

assert len(test_metadata_df) == len(
    sample_submission_df
), (
    "Test metadata and sample submission row counts "
    "do not match."
)

assert not test_metadata_df[
    "id"
].duplicated().any(), (
    "Duplicate Kaggle submission IDs were found."
)

display(
    test_metadata_df.head()
)

print(
    "Kaggle submission IDs attached successfully."
)

,id,well_id,well_order,original_row_position,global_prediction_position
0,000d7d20_1442,000d7d20,0,1442,0
1,000d7d20_1443,000d7d20,0,1443,1
2,000d7d20_1444,000d7d20,0,1444,2
3,000d7d20_1445,000d7d20,0,1445,3
4,000d7d20_1446,000d7d20,0,1446,4


Kaggle submission IDs attached successfully.


#### 9.7 Save Engineered Test Artifacts

In [175]:


ENGINEERED_TEST_FEATURES_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "engineered_test_features.parquet"
)

TEST_METADATA_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "test_prediction_metadata.parquet"
)

TEST_FEATURE_SUMMARY_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "test_feature_summary.csv"
)

X_test_final.to_parquet(
    ENGINEERED_TEST_FEATURES_PATH,
    index=False,
)

test_metadata_df.to_parquet(
    TEST_METADATA_PATH,
    index=False,
)

test_construction_summary_df[
    "construction_seconds"
] = round(
    test_feature_seconds,
    4,
)

test_construction_summary_df.to_csv(
    TEST_FEATURE_SUMMARY_PATH,
    index=False,
)

print(
    "Engineered test features saved to:",
    ENGINEERED_TEST_FEATURES_PATH,
)

print(
    "Test metadata saved to:",
    TEST_METADATA_PATH,
)

print(
    "Test feature summary saved to:",
    TEST_FEATURE_SUMMARY_PATH,
)

Engineered test features saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\engineered_test_features.parquet
Test metadata saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\test_prediction_metadata.parquet
Test feature summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\test_feature_summary.csv


#### 9.8 Section 9 Summary

In [176]:


section_09_summary_df = pd.DataFrame([
    {
        "number_of_test_wells": (
            test_metadata_df[
                "well_id"
            ].nunique()
        ),
        "number_of_prediction_rows": len(
            X_test_final
        ),
        "number_of_model_features": len(
            MODEL_FEATURE_COLUMNS
        ),
        "missing_feature_values": int(
            X_test_final
            .isna()
            .sum()
            .sum()
        ),
        "infinite_feature_values": (
            test_infinite_values
        ),
        "construction_seconds": round(
            test_feature_seconds,
            4,
        ),
        "matches_submission_rows": (
            len(X_test_final)
            == len(sample_submission_df)
        ),
    }
])

display(
    section_09_summary_df
)

print(
    "Section 9 completed successfully."
)

,number_of_test_wells,number_of_prediction_rows,number_of_model_features,missing_feature_values,infinite_feature_values,construction_seconds,matches_submission_rows
0,3,14151,6,17217,0,0.0911,True


Section 9 completed successfully.


### 10. Generate Final Model Predictions

This section generates predictions for the Kaggle test rows using the two final production models trained on the complete training dataset.

Predictions are generated separately for the optimized Extra Trees and XGBoost pipelines. The outputs are validated and saved before constructing the final weighted ensemble.

#### 10.1 Verify Prediction Inputs and Models

In [177]:


assert "X_test_final" in globals(), (
    "X_test_final is not available. "
    "Run Section 9 before generating predictions."
)

assert "test_metadata_df" in globals(), (
    "test_metadata_df is not available. "
    "Run Section 9 before generating predictions."
)

assert "final_extra_trees_pipeline" in globals(), (
    "The final Extra Trees pipeline is not available. "
    "Run Section 8 before generating predictions."
)

assert "final_xgboost_pipeline" in globals(), (
    "The final XGBoost pipeline is not available. "
    "Run Section 8 before generating predictions."
)

assert len(X_test_final) == len(
    test_metadata_df
), (
    "Test feature and metadata row counts do not match."
)

assert list(
    X_test_final.columns
) == MODEL_FEATURE_COLUMNS, (
    "The test feature columns are not in the expected order."
)

print(
    "Prediction rows:",
    f"{len(X_test_final):,}",
)

print(
    "Prediction features:",
    len(X_test_final.columns),
)

print(
    "Extra Trees model:",
    type(
        final_extra_trees_pipeline
        .named_steps["model"]
    ).__name__,
)

print(
    "XGBoost model:",
    type(
        final_xgboost_pipeline
        .named_steps["model"]
    ).__name__,
)

print(
    "Prediction inputs and models verified successfully."
)

Prediction rows: 14,151
Prediction features: 6
Extra Trees model: ExtraTreesRegressor
XGBoost model: XGBRegressor
Prediction inputs and models verified successfully.


#### 10.2 Generate Extra Trees Predictions

In [178]:


extra_trees_prediction_start = time.time()

extra_trees_test_predictions = (
    final_extra_trees_pipeline.predict(
        X_test_final
    )
)

extra_trees_prediction_seconds = (
    time.time()
    - extra_trees_prediction_start
)

extra_trees_test_predictions = np.asarray(
    extra_trees_test_predictions,
    dtype=float,
)

print(
    "Extra Trees predictions generated:",
    len(extra_trees_test_predictions),
)

print(
    "Prediction time:",
    round(
        extra_trees_prediction_seconds,
        4,
    ),
    "seconds",
)

print(
    "Prediction minimum:",
    round(
        float(
            extra_trees_test_predictions.min()
        ),
        4,
    ),
)

print(
    "Prediction maximum:",
    round(
        float(
            extra_trees_test_predictions.max()
        ),
        4,
    ),
)

print(
    "Prediction mean:",
    round(
        float(
            extra_trees_test_predictions.mean()
        ),
        4,
    ),
)

Extra Trees predictions generated: 14151
Prediction time: 0.1358 seconds
Prediction minimum: 11583.2344
Prediction maximum: 12272.3935
Prediction mean: 11905.3891


#### 10.3 Validate Extra Trees Predictions

In [179]:


assert len(
    extra_trees_test_predictions
) == len(
    X_test_final
), (
    "Extra Trees prediction count does not match "
    "the test feature row count."
)

assert np.isfinite(
    extra_trees_test_predictions
).all(), (
    "Extra Trees predictions contain NaN "
    "or infinite values."
)

print(
    "Extra Trees predictions validated successfully."
)

Extra Trees predictions validated successfully.


#### 10.4 Generate XGBoost Predictions

In [180]:


xgboost_prediction_start = time.time()

xgboost_test_predictions = (
    final_xgboost_pipeline.predict(
        X_test_final
    )
)

xgboost_prediction_seconds = (
    time.time()
    - xgboost_prediction_start
)

xgboost_test_predictions = np.asarray(
    xgboost_test_predictions,
    dtype=float,
)

print(
    "XGBoost predictions generated:",
    len(xgboost_test_predictions),
)

print(
    "Prediction time:",
    round(
        xgboost_prediction_seconds,
        4,
    ),
    "seconds",
)

print(
    "Prediction minimum:",
    round(
        float(
            xgboost_test_predictions.min()
        ),
        4,
    ),
)

print(
    "Prediction maximum:",
    round(
        float(
            xgboost_test_predictions.max()
        ),
        4,
    ),
)

print(
    "Prediction mean:",
    round(
        float(
            xgboost_test_predictions.mean()
        ),
        4,
    ),
)

XGBoost predictions generated: 14151
Prediction time: 0.1197 seconds
Prediction minimum: 11535.8369
Prediction maximum: 12258.7939
Prediction mean: 11908.166


#### 10.5 Validate XGBoost Predictions

In [181]:

assert len(
    xgboost_test_predictions
) == len(
    X_test_final
), (
    "XGBoost prediction count does not match "
    "the test feature row count."
)

assert np.isfinite(
    xgboost_test_predictions
).all(), (
    "XGBoost predictions contain NaN "
    "or infinite values."
)

print(
    "XGBoost predictions validated successfully."
)

XGBoost predictions validated successfully.


#### 10.6 Create Individual Prediction Table

In [182]:


individual_test_predictions_df = (
    test_metadata_df.copy()
)

individual_test_predictions_df[
    "extra_trees_prediction"
] = extra_trees_test_predictions

individual_test_predictions_df[
    "xgboost_prediction"
] = xgboost_test_predictions

individual_test_predictions_df[
    "absolute_model_difference"
] = np.abs(
    individual_test_predictions_df[
        "extra_trees_prediction"
    ]
    - individual_test_predictions_df[
        "xgboost_prediction"
    ]
)

individual_test_predictions_df[
    "signed_model_difference"
] = (
    individual_test_predictions_df[
        "extra_trees_prediction"
    ]
    - individual_test_predictions_df[
        "xgboost_prediction"
    ]
)

display(
    individual_test_predictions_df.head()
)

print(
    "Individual prediction table shape:",
    individual_test_predictions_df.shape,
)

,id,well_id,well_order,original_row_position,global_prediction_position,extra_trees_prediction,xgboost_prediction,absolute_model_difference,signed_model_difference
0,000d7d20_1442,000d7d20,0,1442,0,11764.561715,11793.653320,29.091606,-29.091606
1,000d7d20_1443,000d7d20,0,1443,1,11764.561715,11793.653320,29.091606,-29.091606
2,000d7d20_1444,000d7d20,0,1444,2,11764.561715,11793.653320,29.091606,-29.091606
3,000d7d20_1445,000d7d20,0,1445,3,11767.855654,11794.352539,26.496885,-26.496885
4,000d7d20_1446,000d7d20,0,1446,4,11767.826797,11794.352539,26.525742,-26.525742


Individual prediction table shape: (14151, 9)


#### 10.7 Compare Model Predictions

In [183]:


prediction_correlation = float(
    np.corrcoef(
        extra_trees_test_predictions,
        xgboost_test_predictions,
    )[0, 1]
)

prediction_difference_summary_df = (
    pd.DataFrame([
        {
            "metric": "correlation",
            "value": prediction_correlation,
        },
        {
            "metric": "mean_absolute_difference",
            "value": float(
                individual_test_predictions_df[
                    "absolute_model_difference"
                ].mean()
            ),
        },
        {
            "metric": "median_absolute_difference",
            "value": float(
                individual_test_predictions_df[
                    "absolute_model_difference"
                ].median()
            ),
        },
        {
            "metric": "maximum_absolute_difference",
            "value": float(
                individual_test_predictions_df[
                    "absolute_model_difference"
                ].max()
            ),
        },
        {
            "metric": "mean_signed_difference",
            "value": float(
                individual_test_predictions_df[
                    "signed_model_difference"
                ].mean()
            ),
        },
    ])
)

display(
    prediction_difference_summary_df
)

,metric,value
0,correlation,0.998271
1,mean_absolute_difference,12.864658
2,median_absolute_difference,10.373536
3,maximum_absolute_difference,63.577745
4,mean_signed_difference,-2.776944


#### 10.8 Summarize Predictions by Test Well

In [184]:


prediction_summary_by_well_df = (
    individual_test_predictions_df
    .groupby(
        "well_id",
        as_index=False,
    )
    .agg(
        prediction_rows=(
            "id",
            "size",
        ),
        extra_trees_min=(
            "extra_trees_prediction",
            "min",
        ),
        extra_trees_max=(
            "extra_trees_prediction",
            "max",
        ),
        extra_trees_mean=(
            "extra_trees_prediction",
            "mean",
        ),
        extra_trees_std=(
            "extra_trees_prediction",
            "std",
        ),
        xgboost_min=(
            "xgboost_prediction",
            "min",
        ),
        xgboost_max=(
            "xgboost_prediction",
            "max",
        ),
        xgboost_mean=(
            "xgboost_prediction",
            "mean",
        ),
        xgboost_std=(
            "xgboost_prediction",
            "std",
        ),
        mean_absolute_model_difference=(
            "absolute_model_difference",
            "mean",
        ),
        maximum_absolute_model_difference=(
            "absolute_model_difference",
            "max",
        ),
    )
)

display(
    prediction_summary_by_well_df
)

,well_id,prediction_rows,extra_trees_min,extra_trees_max,extra_trees_mean,extra_trees_std,xgboost_min,xgboost_max,xgboost_mean,xgboost_std,mean_absolute_model_difference,maximum_absolute_model_difference
0,000d7d20,3836,11719.131051,11802.792847,11744.877082,11.337349,11703.954102,11830.285156,11751.398070,29.535795,18.383083,59.269890
1,00bbac68,6014,12183.988635,12272.393506,12224.431737,17.609641,12176.901367,12258.793945,12226.930123,17.297336,9.280873,49.415190
2,00e12e8b,4301,11583.234352,11618.225298,11602.436639,8.038412,11535.836914,11665.477539,11602.263824,21.668841,12.953987,63.577745


#### 10.9 Validate Prediction Ordering

In [186]:


assert individual_test_predictions_df[
    "global_prediction_position"
].is_monotonic_increasing, (
    "Global prediction positions are not ordered."
)

expected_positions = np.arange(
    len(individual_test_predictions_df)
)

assert np.array_equal(
    individual_test_predictions_df[
        "global_prediction_position"
    ].to_numpy(),
    expected_positions,
), (
    "Global prediction positions are not continuous."
)

assert np.array_equal(
    individual_test_predictions_df[
        "id"
    ].to_numpy(),
    sample_submission_df[
        "id"
    ].to_numpy(),
), (
    "Prediction IDs are not aligned with "
    "the sample submission."
)

print(
    "Prediction row ordering matches "
    "the sample submission."
)

Prediction row ordering matches the sample submission.


#### 10.10 Save Individual Model Predictions

In [187]:


INDIVIDUAL_TEST_PREDICTIONS_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "individual_test_predictions.parquet"
)

PREDICTION_SUMMARY_BY_WELL_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "prediction_summary_by_well.csv"
)

PREDICTION_DIFFERENCE_SUMMARY_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "prediction_difference_summary.csv"
)

individual_test_predictions_df.to_parquet(
    INDIVIDUAL_TEST_PREDICTIONS_PATH,
    index=False,
)

prediction_summary_by_well_df.to_csv(
    PREDICTION_SUMMARY_BY_WELL_PATH,
    index=False,
)

prediction_difference_summary_df.to_csv(
    PREDICTION_DIFFERENCE_SUMMARY_PATH,
    index=False,
)

print(
    "Individual predictions saved to:",
    INDIVIDUAL_TEST_PREDICTIONS_PATH,
)

print(
    "Per-well prediction summary saved to:",
    PREDICTION_SUMMARY_BY_WELL_PATH,
)

print(
    "Prediction difference summary saved to:",
    PREDICTION_DIFFERENCE_SUMMARY_PATH,
)

Individual predictions saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\individual_test_predictions.parquet
Per-well prediction summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\prediction_summary_by_well.csv
Prediction difference summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\prediction_difference_summary.csv


#### 10.11 Save Prediction Runtime Summary

In [188]:


prediction_runtime_summary_df = pd.DataFrame([
    {
        "model": "Extra Trees Optimized",
        "prediction_rows": len(
            extra_trees_test_predictions
        ),
        "prediction_seconds": round(
            extra_trees_prediction_seconds,
            6,
        ),
        "rows_per_second": round(
            len(extra_trees_test_predictions)
            / extra_trees_prediction_seconds,
            2,
        )
        if extra_trees_prediction_seconds > 0
        else np.nan,
    },
    {
        "model": "XGBoost Optimized",
        "prediction_rows": len(
            xgboost_test_predictions
        ),
        "prediction_seconds": round(
            xgboost_prediction_seconds,
            6,
        ),
        "rows_per_second": round(
            len(xgboost_test_predictions)
            / xgboost_prediction_seconds,
            2,
        )
        if xgboost_prediction_seconds > 0
        else np.nan,
    },
])

PREDICTION_RUNTIME_SUMMARY_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "prediction_runtime_summary.csv"
)

prediction_runtime_summary_df.to_csv(
    PREDICTION_RUNTIME_SUMMARY_PATH,
    index=False,
)

display(
    prediction_runtime_summary_df
)

print(
    "Prediction runtime summary saved to:",
    PREDICTION_RUNTIME_SUMMARY_PATH,
)

,model,prediction_rows,prediction_seconds,rows_per_second
0,Extra Trees Optimized,14151,0.135849,104167.13
1,XGBoost Optimized,14151,0.119678,118242.03


Prediction runtime summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\prediction_runtime_summary.csv


#### 10.12 Section 10 Summary

In [189]:

section_10_summary_df = pd.DataFrame([
    {
        "prediction_rows": len(
            individual_test_predictions_df
        ),
        "number_of_test_wells": (
            individual_test_predictions_df[
                "well_id"
            ].nunique()
        ),
        "extra_trees_predictions_valid": bool(
            np.isfinite(
                extra_trees_test_predictions
            ).all()
        ),
        "xgboost_predictions_valid": bool(
            np.isfinite(
                xgboost_test_predictions
            ).all()
        ),
        "prediction_correlation": round(
            prediction_correlation,
            6,
        ),
        "mean_absolute_model_difference": round(
            float(
                individual_test_predictions_df[
                    "absolute_model_difference"
                ].mean()
            ),
            6,
        ),
        "ids_match_sample_submission": bool(
            np.array_equal(
                individual_test_predictions_df[
                    "id"
                ].to_numpy(),
                sample_submission_df[
                    "id"
                ].to_numpy(),
            )
        ),
    }
])

display(
    section_10_summary_df
)

print(
    "Section 10 completed successfully."
)

,prediction_rows,number_of_test_wells,extra_trees_predictions_valid,xgboost_predictions_valid,prediction_correlation,mean_absolute_model_difference,ids_match_sample_submission
0,14151,3,True,True,0.998271,12.864658,True


Section 10 completed successfully.


### 11. Create the Final Weighted Ensemble

This section combines the optimized Extra Trees and XGBoost predictions using the ensemble weights selected during out-of-fold evaluation in Notebook 07B.

The weighted ensemble was selected because it achieved a lower out-of-fold RMSE than the best individual model.

#### 11.1 Verify Ensemble Weights

In [190]:


assert "EXTRA_TREES_WEIGHT" in globals(), (
    "EXTRA_TREES_WEIGHT is not available."
)

assert "XGBOOST_WEIGHT" in globals(), (
    "XGBOOST_WEIGHT is not available."
)

ensemble_weight_sum = (
    EXTRA_TREES_WEIGHT
    + XGBOOST_WEIGHT
)

assert np.isclose(
    ensemble_weight_sum,
    1.0,
), (
    "The ensemble weights must sum to 1.0.\n"
    f"Current sum: {ensemble_weight_sum}"
)

assert EXTRA_TREES_WEIGHT >= 0, (
    "Extra Trees ensemble weight cannot be negative."
)

assert XGBOOST_WEIGHT >= 0, (
    "XGBoost ensemble weight cannot be negative."
)

print(
    "Extra Trees weight:",
    EXTRA_TREES_WEIGHT,
)

print(
    "XGBoost weight:",
    XGBOOST_WEIGHT,
)

print(
    "Weight sum:",
    ensemble_weight_sum,
)

print(
    "Ensemble weights validated successfully."
)

Extra Trees weight: 0.8
XGBoost weight: 0.2
Weight sum: 1.0
Ensemble weights validated successfully.


#### 11.2 Generate Weighted Ensemble Predictions

In [191]:

ensemble_prediction_start = time.time()

weighted_ensemble_predictions = (
    EXTRA_TREES_WEIGHT
    * extra_trees_test_predictions
    +
    XGBOOST_WEIGHT
    * xgboost_test_predictions
)

ensemble_prediction_seconds = (
    time.time()
    - ensemble_prediction_start
)

weighted_ensemble_predictions = np.asarray(
    weighted_ensemble_predictions,
    dtype=float,
)

print(
    "Weighted ensemble predictions generated:",
    len(weighted_ensemble_predictions),
)

print(
    "Ensemble calculation time:",
    round(
        ensemble_prediction_seconds,
        6,
    ),
    "seconds",
)

Weighted ensemble predictions generated: 14151
Ensemble calculation time: 0.000809 seconds


#### 11.3 Validate Ensemble Predictions

In [192]:


assert len(
    weighted_ensemble_predictions
) == len(
    X_test_final
), (
    "Ensemble prediction count does not match "
    "the test feature row count."
)

assert np.isfinite(
    weighted_ensemble_predictions
).all(), (
    "The ensemble predictions contain NaN "
    "or infinite values."
)

assert len(
    weighted_ensemble_predictions
) == len(
    sample_submission_df
), (
    "The ensemble prediction count does not match "
    "the sample submission."
)

print(
    "Prediction minimum:",
    round(
        float(
            weighted_ensemble_predictions.min()
        ),
        4,
    ),
)

print(
    "Prediction maximum:",
    round(
        float(
            weighted_ensemble_predictions.max()
        ),
        4,
    ),
)

print(
    "Prediction mean:",
    round(
        float(
            weighted_ensemble_predictions.mean()
        ),
        4,
    ),
)

print(
    "Prediction median:",
    round(
        float(
            np.median(
                weighted_ensemble_predictions
            )
        ),
        4,
    ),
)

print(
    "Prediction standard deviation:",
    round(
        float(
            weighted_ensemble_predictions.std()
        ),
        4,
    ),
)

print(
    "Weighted ensemble predictions validated successfully."
)

Prediction minimum: 11577.2186
Prediction maximum: 12268.2152
Prediction mean: 11905.9445
Prediction median: 11755.8804
Prediction standard deviation: 279.9543
Weighted ensemble predictions validated successfully.


#### 11.4 Verify Convex Ensemble Bounds

In [193]:


individual_prediction_minimum = np.minimum(
    extra_trees_test_predictions,
    xgboost_test_predictions,
)

individual_prediction_maximum = np.maximum(
    extra_trees_test_predictions,
    xgboost_test_predictions,
)

lower_bound_violations = int(
    (
        weighted_ensemble_predictions
        < individual_prediction_minimum
        - 1e-10
    ).sum()
)

upper_bound_violations = int(
    (
        weighted_ensemble_predictions
        > individual_prediction_maximum
        + 1e-10
    ).sum()
)

assert lower_bound_violations == 0, (
    "Some ensemble predictions are below both "
    "individual model predictions."
)

assert upper_bound_violations == 0, (
    "Some ensemble predictions are above both "
    "individual model predictions."
)

print(
    "Lower-bound violations:",
    lower_bound_violations,
)

print(
    "Upper-bound violations:",
    upper_bound_violations,
)

print(
    "All ensemble predictions lie between "
    "the individual model predictions."
)

Lower-bound violations: 0
Upper-bound violations: 0
All ensemble predictions lie between the individual model predictions.


#### 11.5 Create Final Prediction Table

In [195]:


final_test_predictions_df = (
    individual_test_predictions_df.copy()
)

final_test_predictions_df[
    "extra_trees_weight"
] = EXTRA_TREES_WEIGHT

final_test_predictions_df[
    "xgboost_weight"
] = XGBOOST_WEIGHT

final_test_predictions_df[
    "weighted_ensemble_prediction"
] = weighted_ensemble_predictions

final_test_predictions_df[
    "ensemble_minus_extra_trees"
] = (
    final_test_predictions_df[
        "weighted_ensemble_prediction"
    ]
    - final_test_predictions_df[
        "extra_trees_prediction"
    ]
)

final_test_predictions_df[
    "ensemble_minus_xgboost"
] = (
    final_test_predictions_df[
        "weighted_ensemble_prediction"
    ]
    - final_test_predictions_df[
        "xgboost_prediction"
    ]
)

display(
    final_test_predictions_df.head()
)

print(
    "Final prediction table shape:",
    final_test_predictions_df.shape,
)

,id,well_id,well_order,original_row_position,global_prediction_position,extra_trees_prediction,xgboost_prediction,absolute_model_difference,signed_model_difference,extra_trees_weight,xgboost_weight,weighted_ensemble_prediction,ensemble_minus_extra_trees,ensemble_minus_xgboost
0,000d7d20_1442,000d7d20,0,1442,0,11764.561715,11793.653320,29.091606,-29.091606,0.8,0.2,11770.380036,5.818321,-23.273284
1,000d7d20_1443,000d7d20,0,1443,1,11764.561715,11793.653320,29.091606,-29.091606,0.8,0.2,11770.380036,5.818321,-23.273284
2,000d7d20_1444,000d7d20,0,1444,2,11764.561715,11793.653320,29.091606,-29.091606,0.8,0.2,11770.380036,5.818321,-23.273284
3,000d7d20_1445,000d7d20,0,1445,3,11767.855654,11794.352539,26.496885,-26.496885,0.8,0.2,11773.155031,5.299377,-21.197508
4,000d7d20_1446,000d7d20,0,1446,4,11767.826797,11794.352539,26.525742,-26.525742,0.8,0.2,11773.131945,5.305148,-21.220594


Final prediction table shape: (14151, 14)


#### 11.6 Summarize Ensemble Predictions by Well

In [196]:


ensemble_summary_by_well_df = (
    final_test_predictions_df
    .groupby(
        "well_id",
        as_index=False,
    )
    .agg(
        prediction_rows=(
            "id",
            "size",
        ),
        ensemble_min=(
            "weighted_ensemble_prediction",
            "min",
        ),
        ensemble_max=(
            "weighted_ensemble_prediction",
            "max",
        ),
        ensemble_mean=(
            "weighted_ensemble_prediction",
            "mean",
        ),
        ensemble_median=(
            "weighted_ensemble_prediction",
            "median",
        ),
        ensemble_std=(
            "weighted_ensemble_prediction",
            "std",
        ),
        extra_trees_mean=(
            "extra_trees_prediction",
            "mean",
        ),
        xgboost_mean=(
            "xgboost_prediction",
            "mean",
        ),
        mean_model_disagreement=(
            "absolute_model_difference",
            "mean",
        ),
        maximum_model_disagreement=(
            "absolute_model_difference",
            "max",
        ),
    )
)

display(
    ensemble_summary_by_well_df
)

,well_id,prediction_rows,ensemble_min,ensemble_max,ensemble_mean,ensemble_median,ensemble_std,extra_trees_mean,xgboost_mean,mean_model_disagreement,maximum_model_disagreement
0,000d7d20,3836,11716.677302,11808.291309,11746.181280,11752.275145,14.591099,11744.877082,11751.398070,18.383083,59.269890
1,00bbac68,6014,12183.964345,12268.215196,12224.931414,12232.328197,16.907499,12224.431737,12226.930123,9.280873,49.415190
2,00e12e8b,4301,11577.218592,11624.246254,11602.402076,11600.675626,9.689621,11602.436639,11602.263824,12.953987,63.577745


#### 11.7 Inspect Largest Model Disagreements

In [197]:


largest_test_disagreements_df = (
    final_test_predictions_df
    .nlargest(
        20,
        "absolute_model_difference",
    )
    [
        [
            "id",
            "well_id",
            "original_row_position",
            "extra_trees_prediction",
            "xgboost_prediction",
            "weighted_ensemble_prediction",
            "absolute_model_difference",
        ]
    ]
    .reset_index(drop=True)
)

display(
    largest_test_disagreements_df
)

,id,well_id,original_row_position,extra_trees_prediction,xgboost_prediction,weighted_ensemble_prediction,absolute_model_difference
0,00e12e8b_5049,00e12e8b,5049,11601.899794,11665.477539,11614.615343,63.577745
1,00e12e8b_5047,00e12e8b,5047,11601.918028,11665.477539,11614.629930,63.559511
2,00e12e8b_5051,00e12e8b,5051,11601.928093,11665.477539,11614.637983,63.549446
3,00e12e8b_5055,00e12e8b,5055,11601.934263,11665.477539,11614.642918,63.543276
4,00e12e8b_5048,00e12e8b,5048,11601.949566,11665.477539,11614.655160,63.527974
5,00e12e8b_5045,00e12e8b,5045,11601.983454,11665.477539,11614.682271,63.494085
6,00e12e8b_5050,00e12e8b,5050,11601.990733,11665.477539,11614.688094,63.486806
7,00e12e8b_5046,00e12e8b,5046,11602.015428,11665.477539,11614.707850,63.462111
8,00e12e8b_5040,00e12e8b,5040,11602.046094,11664.249023,11614.486680,62.202930
9,00e12e8b_5041,00e12e8b,5041,11602.046094,11664.249023,11614.486680,62.202930


#### 11.8 Save Final Ensemble Artifacts

In [198]:


FINAL_TEST_PREDICTIONS_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "final_test_predictions.parquet"
)

ENSEMBLE_SUMMARY_BY_WELL_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "ensemble_summary_by_well.csv"
)

LARGEST_TEST_DISAGREEMENTS_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "largest_test_model_disagreements.csv"
)

final_test_predictions_df.to_parquet(
    FINAL_TEST_PREDICTIONS_PATH,
    index=False,
)

ensemble_summary_by_well_df.to_csv(
    ENSEMBLE_SUMMARY_BY_WELL_PATH,
    index=False,
)

largest_test_disagreements_df.to_csv(
    LARGEST_TEST_DISAGREEMENTS_PATH,
    index=False,
)

print(
    "Final test predictions saved to:",
    FINAL_TEST_PREDICTIONS_PATH,
)

print(
    "Per-well ensemble summary saved to:",
    ENSEMBLE_SUMMARY_BY_WELL_PATH,
)

print(
    "Largest model disagreements saved to:",
    LARGEST_TEST_DISAGREEMENTS_PATH,
)

Final test predictions saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\final_test_predictions.parquet
Per-well ensemble summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\ensemble_summary_by_well.csv
Largest model disagreements saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\largest_test_model_disagreements.csv


#### 11.9 Save Ensemble Metadata

In [199]:


ensemble_metadata = {
    "candidate_type": "ensemble",
    "candidate_name": "Weighted Ensemble",
    "model_a": "Extra Trees Optimized",
    "model_b": "XGBoost Optimized",
    "weight_model_a": float(
        EXTRA_TREES_WEIGHT
    ),
    "weight_model_b": float(
        XGBOOST_WEIGHT
    ),
    "number_of_prediction_rows": int(
        len(
            weighted_ensemble_predictions
        )
    ),
    "number_of_test_wells": int(
        final_test_predictions_df[
            "well_id"
        ].nunique()
    ),
    "prediction_minimum": float(
        weighted_ensemble_predictions.min()
    ),
    "prediction_maximum": float(
        weighted_ensemble_predictions.max()
    ),
    "prediction_mean": float(
        weighted_ensemble_predictions.mean()
    ),
    "prediction_standard_deviation": float(
        weighted_ensemble_predictions.std()
    ),
    "ensemble_calculation_seconds": float(
        ensemble_prediction_seconds
    ),
    "source_selection_notebook": (
        "Notebook 07B"
    ),
    "training_notebook": (
        "Notebook 08"
    ),
}

ENSEMBLE_METADATA_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "final_ensemble_metadata.json"
)

with open(
    ENSEMBLE_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        ensemble_metadata,
        file,
        indent=4,
    )

print(
    "Ensemble metadata saved to:",
    ENSEMBLE_METADATA_PATH,
)

Ensemble metadata saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\final_ensemble_metadata.json


#### 11.10 Section 11 Summary

In [200]:


section_11_summary_df = pd.DataFrame([
    {
        "candidate_name": (
            "Weighted Ensemble"
        ),
        "extra_trees_weight": (
            EXTRA_TREES_WEIGHT
        ),
        "xgboost_weight": (
            XGBOOST_WEIGHT
        ),
        "weight_sum": (
            ensemble_weight_sum
        ),
        "prediction_rows": len(
            weighted_ensemble_predictions
        ),
        "number_of_test_wells": (
            final_test_predictions_df[
                "well_id"
            ].nunique()
        ),
        "all_predictions_finite": bool(
            np.isfinite(
                weighted_ensemble_predictions
            ).all()
        ),
        "lower_bound_violations": (
            lower_bound_violations
        ),
        "upper_bound_violations": (
            upper_bound_violations
        ),
        "prediction_minimum": float(
            weighted_ensemble_predictions.min()
        ),
        "prediction_maximum": float(
            weighted_ensemble_predictions.max()
        ),
        "prediction_mean": float(
            weighted_ensemble_predictions.mean()
        ),
    }
])

display(
    section_11_summary_df
)

print(
    "Section 11 completed successfully."
)

,candidate_name,extra_trees_weight,xgboost_weight,weight_sum,prediction_rows,number_of_test_wells,all_predictions_finite,lower_bound_violations,upper_bound_violations,prediction_minimum,prediction_maximum,prediction_mean
0,Weighted Ensemble,0.8,0.2,1.0,14151,3,True,0,0,11577.218592,12268.215196,11905.944473


Section 11 completed successfully.


### 12. Create and Validate the Kaggle Submission File

This section inserts the final weighted ensemble predictions into the Kaggle sample submission structure.

The completed submission is validated for column names, row count, ID order, missing values, duplicate IDs, and numerical validity before being saved as `submission.csv`.

#### 12.1 Create Final Submission DataFrame

In [201]:


submission_df = (
    sample_submission_df.copy()
)

assert "id" in submission_df.columns, (
    "The sample submission does not contain an 'id' column."
)

assert "tvt" in submission_df.columns, (
    "The sample submission does not contain a 'tvt' column."
)

assert len(
    weighted_ensemble_predictions
) == len(
    submission_df
), (
    "Prediction count does not match the sample submission."
)

submission_df[
    "tvt"
] = weighted_ensemble_predictions

display(
    submission_df.head()
)

print(
    "Submission shape:",
    submission_df.shape,
)

,id,tvt
0,000d7d20_1442,11770.380036
1,000d7d20_1443,11770.380036
2,000d7d20_1444,11770.380036
3,000d7d20_1445,11773.155031
4,000d7d20_1446,11773.131945


Submission shape: (14151, 2)


#### 12.2 Validate Submission Schema

In [202]:


expected_submission_columns = [
    "id",
    "tvt",
]

assert list(
    submission_df.columns
) == expected_submission_columns, (
    "Submission columns do not match the required schema.\n"
    f"Expected: {expected_submission_columns}\n"
    f"Actual: {submission_df.columns.tolist()}"
)

assert len(
    submission_df
) == len(
    sample_submission_df
), (
    "Submission row count does not match "
    "the sample submission."
)

assert np.array_equal(
    submission_df["id"].to_numpy(),
    sample_submission_df["id"].to_numpy(),
), (
    "Submission IDs are not in the same order "
    "as the sample submission."
)

print(
    "Submission schema validated successfully."
)

Submission schema validated successfully.


#### 12.3 Validate Submission Values

In [203]:


duplicate_id_count = int(
    submission_df[
        "id"
    ].duplicated().sum()
)

missing_id_count = int(
    submission_df[
        "id"
    ].isna().sum()
)

missing_prediction_count = int(
    submission_df[
        "tvt"
    ].isna().sum()
)

infinite_prediction_count = int(
    np.isinf(
        submission_df[
            "tvt"
        ].to_numpy(
            dtype=float
        )
    ).sum()
)

assert duplicate_id_count == 0, (
    "The submission contains duplicate IDs."
)

assert missing_id_count == 0, (
    "The submission contains missing IDs."
)

assert missing_prediction_count == 0, (
    "The submission contains missing predictions."
)

assert infinite_prediction_count == 0, (
    "The submission contains infinite predictions."
)

print(
    "Duplicate IDs:",
    duplicate_id_count,
)

print(
    "Missing IDs:",
    missing_id_count,
)

print(
    "Missing predictions:",
    missing_prediction_count,
)

print(
    "Infinite predictions:",
    infinite_prediction_count,
)

print(
    "Submission values validated successfully."
)

Duplicate IDs: 0
Missing IDs: 0
Missing predictions: 0
Infinite predictions: 0
Submission values validated successfully.


#### 12.4 Review Submission Statistics

In [204]:


submission_statistics_df = pd.DataFrame([
    {
        "row_count": len(
            submission_df
        ),
        "unique_ids": submission_df[
            "id"
        ].nunique(),
        "prediction_minimum": float(
            submission_df[
                "tvt"
            ].min()
        ),
        "prediction_maximum": float(
            submission_df[
                "tvt"
            ].max()
        ),
        "prediction_mean": float(
            submission_df[
                "tvt"
            ].mean()
        ),
        "prediction_median": float(
            submission_df[
                "tvt"
            ].median()
        ),
        "prediction_standard_deviation": float(
            submission_df[
                "tvt"
            ].std()
        ),
    }
])

display(
    submission_statistics_df
)

,row_count,unique_ids,prediction_minimum,prediction_maximum,prediction_mean,prediction_median,prediction_standard_deviation
0,14151,14151,11577.218592,12268.215196,11905.944473,11755.880414,279.964189


#### 12.5 Confirm Prediction Alignment

In [205]:


assert np.allclose(
    submission_df[
        "tvt"
    ].to_numpy(
        dtype=float
    ),
    weighted_ensemble_predictions,
), (
    "Submission predictions do not match "
    "the final ensemble predictions."
)

assert np.array_equal(
    submission_df[
        "id"
    ].to_numpy(),
    test_metadata_df[
        "id"
    ].to_numpy(),
), (
    "Submission IDs do not match "
    "the test prediction metadata."
)

print(
    "Submission IDs and predictions are correctly aligned."
)

Submission IDs and predictions are correctly aligned.


#### 12.6 Configure Submission Output Directory

In [206]:


SUBMISSIONS_DIR = (
    PROJECT_ROOT
    / "submissions"
)

SUBMISSIONS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FINAL_SUBMISSION_PATH = (
    SUBMISSIONS_DIR
    / "submission.csv"
)

NOTEBOOK_08_SUBMISSION_COPY_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "submission.csv"
)

print(
    "Submission directory:",
    SUBMISSIONS_DIR,
)

print(
    "Final submission path:",
    FINAL_SUBMISSION_PATH,
)

Submission directory: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\submissions
Final submission path: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\submissions\submission.csv


#### 12.7 Save Final Submission

In [207]:


submission_df.to_csv(
    FINAL_SUBMISSION_PATH,
    index=False,
)

submission_df.to_csv(
    NOTEBOOK_08_SUBMISSION_COPY_PATH,
    index=False,
)

print(
    "Final Kaggle submission saved to:",
    FINAL_SUBMISSION_PATH,
)

print(
    "Notebook results copy saved to:",
    NOTEBOOK_08_SUBMISSION_COPY_PATH,
)

Final Kaggle submission saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\submissions\submission.csv
Notebook results copy saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\submission.csv


#### 12.8 Reload and Verify Saved Submission

In [208]:


saved_submission_df = pd.read_csv(
    FINAL_SUBMISSION_PATH
)

assert list(
    saved_submission_df.columns
) == [
    "id",
    "tvt",
], (
    "The saved submission has incorrect columns."
)

assert len(
    saved_submission_df
) == len(
    sample_submission_df
), (
    "The saved submission has an incorrect row count."
)

assert np.array_equal(
    saved_submission_df[
        "id"
    ].to_numpy(),
    sample_submission_df[
        "id"
    ].to_numpy(),
), (
    "The saved submission IDs are incorrectly ordered."
)

assert np.allclose(
    saved_submission_df[
        "tvt"
    ].to_numpy(
        dtype=float
    ),
    weighted_ensemble_predictions,
), (
    "The saved submission predictions differ "
    "from the final ensemble predictions."
)

assert saved_submission_df[
    "tvt"
].notna().all(), (
    "The saved submission contains missing predictions."
)

assert np.isfinite(
    saved_submission_df[
        "tvt"
    ].to_numpy(
        dtype=float
    )
).all(), (
    "The saved submission contains invalid predictions."
)

print(
    "Saved submission reloaded and verified successfully."
)

Saved submission reloaded and verified successfully.


#### 12.9 Preview Final Submission

In [209]:


print("First five rows:")

display(
    saved_submission_df.head()
)

print("Last five rows:")

display(
    saved_submission_df.tail()
)

First five rows:


,id,tvt
0,000d7d20_1442,11770.380036
1,000d7d20_1443,11770.380036
2,000d7d20_1444,11770.380036
3,000d7d20_1445,11773.155031
4,000d7d20_1446,11773.131945


Last five rows:


,id,tvt
14146,00e12e8b_6379,11594.628832
14147,00e12e8b_6380,11594.571458
14148,00e12e8b_6381,11594.567880
14149,00e12e8b_6382,11579.545448
14150,00e12e8b_6383,11594.572859


#### 12.10 Save Submission Metadata

In [210]:


submission_metadata = {
    "submission_filename": (
        FINAL_SUBMISSION_PATH.name
    ),
    "submission_path": str(
        FINAL_SUBMISSION_PATH
    ),
    "number_of_rows": int(
        len(saved_submission_df)
    ),
    "number_of_columns": int(
        saved_submission_df.shape[1]
    ),
    "columns": (
        saved_submission_df.columns.tolist()
    ),
    "prediction_column": "tvt",
    "candidate_name": "Weighted Ensemble",
    "extra_trees_weight": float(
        EXTRA_TREES_WEIGHT
    ),
    "xgboost_weight": float(
        XGBOOST_WEIGHT
    ),
    "prediction_minimum": float(
        saved_submission_df[
            "tvt"
        ].min()
    ),
    "prediction_maximum": float(
        saved_submission_df[
            "tvt"
        ].max()
    ),
    "prediction_mean": float(
        saved_submission_df[
            "tvt"
        ].mean()
    ),
    "all_predictions_finite": bool(
        np.isfinite(
            saved_submission_df[
                "tvt"
            ].to_numpy(
                dtype=float
            )
        ).all()
    ),
    "ids_match_sample_submission": bool(
        np.array_equal(
            saved_submission_df[
                "id"
            ].to_numpy(),
            sample_submission_df[
                "id"
            ].to_numpy(),
        )
    ),
}

SUBMISSION_METADATA_PATH = (
    NOTEBOOK_08_RESULTS_DIR
    / "submission_metadata.json"
)

with open(
    SUBMISSION_METADATA_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        submission_metadata,
        file,
        indent=4,
    )

print(
    "Submission metadata saved to:",
    SUBMISSION_METADATA_PATH,
)

Submission metadata saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\submission_metadata.json


#### 12.11 Section Summary

In [211]:


section_12_summary_df = pd.DataFrame([
    {
        "submission_file": str(
            FINAL_SUBMISSION_PATH
        ),
        "row_count": len(
            saved_submission_df
        ),
        "column_count": (
            saved_submission_df.shape[1]
        ),
        "duplicate_ids": int(
            saved_submission_df[
                "id"
            ].duplicated().sum()
        ),
        "missing_predictions": int(
            saved_submission_df[
                "tvt"
            ].isna().sum()
        ),
        "all_predictions_finite": bool(
            np.isfinite(
                saved_submission_df[
                    "tvt"
                ].to_numpy(
                    dtype=float
                )
            ).all()
        ),
        "ids_match_sample_submission": bool(
            np.array_equal(
                saved_submission_df[
                    "id"
                ].to_numpy(),
                sample_submission_df[
                    "id"
                ].to_numpy(),
            )
        ),
        "prediction_minimum": float(
            saved_submission_df[
                "tvt"
            ].min()
        ),
        "prediction_maximum": float(
            saved_submission_df[
                "tvt"
            ].max()
        ),
    }
])

display(
    section_12_summary_df
)

print(
    "Section 12 completed successfully."
)

print(
    "\nSubmission ready for Kaggle:"
)

print(
    FINAL_SUBMISSION_PATH
)

,submission_file,row_count,column_count,duplicate_ids,missing_predictions,all_predictions_finite,ids_match_sample_submission,prediction_minimum,prediction_maximum
0,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,14151,2,0,0,True,True,11577.218592,12268.215196


Section 12 completed successfully.

Submission ready for Kaggle:
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\submissions\submission.csv


### 13. Save Final Artifacts and Reproducibility Summary

This section verifies that all important Notebook 08 artifacts were generated successfully.

A final inventory of models, prediction files, metadata, and the Kaggle submission is created to ensure the entire inference pipeline is reproducible.

#### 13.1 Verify Final Notebook Artifacts

In [212]:


expected_artifacts = {
    "Final Extra Trees Model":
        FINAL_EXTRA_TREES_MODEL_PATH,

    "Final XGBoost Model":
        FINAL_XGBOOST_MODEL_PATH,

    "Engineered Test Features":
        ENGINEERED_TEST_FEATURES_PATH,

    "Test Metadata":
        TEST_METADATA_PATH,

    "Individual Predictions":
        INDIVIDUAL_TEST_PREDICTIONS_PATH,

    "Final Ensemble Predictions":
        FINAL_TEST_PREDICTIONS_PATH,

    "Submission":
        FINAL_SUBMISSION_PATH,

    "Submission Metadata":
        SUBMISSION_METADATA_PATH,

    "Ensemble Metadata":
        ENSEMBLE_METADATA_PATH,

    "Training Summary":
        final_training_summary_path,

    "Prediction Runtime Summary":
        PREDICTION_RUNTIME_SUMMARY_PATH,
}

artifact_rows = []

for artifact_name, artifact_path in expected_artifacts.items():

    exists = artifact_path.exists()

    artifact_rows.append({
        "artifact": artifact_name,
        "path": str(artifact_path),
        "exists": exists,
        "size_mb": (
            round(
                artifact_path.stat().st_size
                / (1024 ** 2),
                4,
            )
            if exists
            else np.nan
        ),
    })

artifact_inventory_df = pd.DataFrame(
    artifact_rows
)

display(
    artifact_inventory_df
)

assert artifact_inventory_df[
    "exists"
].all(), (
    "One or more notebook artifacts were not created."
)

print(
    "All expected Notebook 08 artifacts were verified."
)

,artifact,path,exists,size_mb
0,Final Extra Trees Model,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,290.0853
1,Final XGBoost Model,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,7.5987
2,Engineered Test Features,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.3426
3,Test Metadata,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.2169
4,Individual Predictions,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.5853
5,Final Ensemble Predictions,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.9278
6,Submission,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.4561
7,Submission Metadata,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.0006
8,Ensemble Metadata,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.0006
9,Training Summary,c:\Dev\Rogii Wellbore Geology\rogii-wellbore-g...,True,0.0004


All expected Notebook 08 artifacts were verified.


#### 13.2 Final Project Summary

In [213]:

project_summary_df = pd.DataFrame([
    {
        "training_wells": training_groups.nunique(),
        "training_rows": len(X_final),
        "test_wells": test_metadata_df[
            "well_id"
        ].nunique(),
        "prediction_rows": len(X_test_final),
        "model_features": len(
            MODEL_FEATURE_COLUMNS
        ),
        "ensemble_weight_extra_trees":
            EXTRA_TREES_WEIGHT,
        "ensemble_weight_xgboost":
            XGBOOST_WEIGHT,
    }
])

display(
    project_summary_df
)

,training_wells,training_rows,test_wells,prediction_rows,model_features,ensemble_weight_extra_trees,ensemble_weight_xgboost
0,773,5092255,3,14151,6,0.8,0.2


#### 13.3 Runtime Summary

In [214]:


runtime_summary_df = pd.DataFrame([
    {
        "step": "Train Extra Trees",
        "seconds": round(
            extra_trees_training_seconds,
            2,
        ),
    },
    {
        "step": "Train XGBoost",
        "seconds": round(
            xgboost_training_seconds,
            2,
        ),
    },
    {
        "step": "Construct Test Features",
        "seconds": round(
            test_feature_seconds,
            2,
        ),
    },
    {
        "step": "Extra Trees Prediction",
        "seconds": round(
            extra_trees_prediction_seconds,
            4,
        ),
    },
    {
        "step": "XGBoost Prediction",
        "seconds": round(
            xgboost_prediction_seconds,
            4,
        ),
    },
    {
        "step": "Weighted Ensemble",
        "seconds": round(
            ensemble_prediction_seconds,
            6,
        ),
    },
])

display(
    runtime_summary_df
)

runtime_summary_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "runtime_summary.csv"
)

runtime_summary_df.to_csv(
    runtime_summary_path,
    index=False,
)

print(
    "Runtime summary saved to:",
    runtime_summary_path,
)

,step,seconds
0,Train Extra Trees,99.250000
1,Train XGBoost,197.370000
2,Construct Test Features,0.090000
3,Extra Trees Prediction,0.135800
4,XGBoost Prediction,0.119700
5,Weighted Ensemble,0.000809


Runtime summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\runtime_summary.csv


#### 13.4 Reproducibility Information

In [215]:


reproducibility_df = pd.DataFrame([
    {
        "item": "Random Seed",
        "value": 42,
    },
    {
        "item": "Final Candidate",
        "value": "Weighted Ensemble",
    },
    {
        "item": "Model A",
        "value": "Extra Trees Optimized",
    },
    {
        "item": "Model B",
        "value": "XGBoost Optimized",
    },
    {
        "item": "Model A Weight",
        "value": EXTRA_TREES_WEIGHT,
    },
    {
        "item": "Model B Weight",
        "value": XGBOOST_WEIGHT,
    },
    {
        "item": "Training Wells",
        "value": training_groups.nunique(),
    },
    {
        "item": "Training Rows",
        "value": len(X_final),
    },
    {
        "item": "Prediction Rows",
        "value": len(X_test_final),
    },
    {
        "item": "Feature Count",
        "value": len(MODEL_FEATURE_COLUMNS),
    },
])

display(
    reproducibility_df
)

reproducibility_path = (
    NOTEBOOK_08_RESULTS_DIR
    / "reproducibility_summary.csv"
)

reproducibility_df.to_csv(
    reproducibility_path,
    index=False,
)

print(
    "Reproducibility summary saved to:",
    reproducibility_path,
)

,item,value
0,Random Seed,42
1,Final Candidate,Weighted Ensemble
2,Model A,Extra Trees Optimized
3,Model B,XGBoost Optimized
4,Model A Weight,0.8
5,Model B Weight,0.2
6,Training Wells,773
7,Training Rows,5092255
8,Prediction Rows,14151
9,Feature Count,6


Reproducibility summary saved to: c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08\reproducibility_summary.csv


#### 13.5 Final Artifact Overview

In [216]:


print("=" * 70)
print("NOTEBOOK 08 COMPLETED SUCCESSFULLY")
print("=" * 70)

print()

print("Final Kaggle Submission:")
print(FINAL_SUBMISSION_PATH)

print()

print("Notebook Results Directory:")
print(NOTEBOOK_08_RESULTS_DIR)

print()

print("Final Models:")
print(FINAL_EXTRA_TREES_MODEL_PATH)
print(FINAL_XGBOOST_MODEL_PATH)

print()

print("Prediction Rows:", len(submission_df))
print("Training Rows:", len(X_final))

print()

print("Selected Ensemble:")
print(
    f"Extra Trees ({EXTRA_TREES_WEIGHT:.2f}) + "
    f"XGBoost ({XGBOOST_WEIGHT:.2f})"
)

print()

print("Notebook 08 finished successfully.")

NOTEBOOK 08 COMPLETED SUCCESSFULLY

Final Kaggle Submission:
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\submissions\submission.csv

Notebook Results Directory:
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\results\notebook_08

Final Models:
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\models\notebook_08\final_extra_trees_pipeline.joblib
c:\Dev\Rogii Wellbore Geology\rogii-wellbore-geology\models\notebook_08\final_xgboost_pipeline.joblib

Prediction Rows: 14151
Training Rows: 5092255

Selected Ensemble:
Extra Trees (0.80) + XGBoost (0.20)

Notebook 08 finished successfully.
